In [ ]:
import os
base = '/kaggle/working/onda'
os.makedirs(os.path.join(base, 'tests'), exist_ok=True)
os.makedirs(os.path.join(base, 'prototypes/wave_mem'), exist_ok=True)
os.makedirs(os.path.join(base, 'prototypes/delta_forget'), exist_ok=True)
os.makedirs(os.path.join(base, 'prototypes/delta_nlms'), exist_ok=True)
os.makedirs(os.path.join(base, 'outputs/n1_wave_complex/cache'), exist_ok=True)
os.makedirs(os.path.join(base, 'outputs/n1_wave_re/cache'), exist_ok=True)


In [ ]:
import os
print('cwd:', os.getcwd())
print('input dirs:', sorted(os.listdir('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'NO INPUT')
for d in sorted(os.listdir('/kaggle/input') or []):
    p = os.path.join('/kaggle/input', d)
    if os.path.isdir(p):
        print('  input', d, '->', sorted(os.listdir(p))[:8])


In [ ]:
# ==== CODIGO ONDA ====
# --- tests/common_smoke.py ---
with open(os.path.join(base, 'tests/common_smoke.py'), 'w', encoding='utf-8') as f:
    f.write('''import random\nimport os\nimport time\nfrom dataclasses import dataclass, asdict\nfrom typing import Dict, List, Tuple, Optional, Callable\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\n\n\nPAD_ID = 0\nBOS_ID = 1\nEOS_ID = 2\nSEP_ID = 3\nQUERY_ID = 4\nASSIGN_ID = 5\nPICK_ID = 6  # legacy (no usado en tests; src/ reason_smoke lo usa)\nFORGET_ID = 6  # alias PICK_ID: id libre rehusado como marker FORGET\nGOTO_ID = 7\nANSWER_ID = 8\nRESERVED_SPECIAL = 9\n\n\nVOCAB_MQAR_SIZE = 89\nN_KEYS_MQAR = 40\nN_VALS_MQAR = 40\nK_OFFSET_MQAR = RESERVED_SPECIAL\nV_OFFSET_MQAR = RESERVED_SPECIAL + N_KEYS_MQAR\n\nVOCAB_COPY_SIZE = 19\nN_DIGITS_COPY = 10\nDIGIT_OFFSET = RESERVED_SPECIAL\n\nVOCAB_ST_SIZE = 17\nN_ENTITIES = 5\nN_LOCATIONS = 3\nENT_OFFSET = RESERVED_SPECIAL\nLOC_OFFSET = RESERVED_SPECIAL + N_ENTITIES\n\n# Dyck-2 (S81): 2 tipos de parens (A/B). Tarea de cierre del top del stack\n# (Suzgun et al. 2019): el prefijo es una secuencia parcialmente balanceada\n# que termina con pila NO vacia; la respuesta es el cierre que toca\n# (CLOSE del tipo del top). Anticorto de \'copiar el ultimo token\': el\n# ultimo token puede ser un CLOSE interior, no el par del top; y\n# n_opened >= 2 fuerza a rastrear el stack completo, no solo el ultimo.\nVOCAB_DYCK_SIZE = 16\nN_DYCK_TYPES = 2\nDYCK_OPEN_BASE = RESERVED_SPECIAL  # 9, 10 (OPEN A/B)\nDYCK_CLOSE_BASE = RESERVED_SPECIAL + N_DYCK_TYPES  # 11, 12 (CLOSE A/B)\n\n# forget_retrieval reusa el vocabulario MQAR (89 tokens). Mismas keys/vals:\n# store de n_pairs pares k,v + instruccion FORGET k_i + QUERY k_j (j != i)\n# + ANSWER v_j. Tests si el modelo sabe descartar par borrado y aun\n# recuperar par presente (GDN acumula, no borra; s4d_carve tiene erase\n# interno pero no integrado a una task).\nVOCAB_FORGET_SIZE = VOCAB_MQAR_SIZE\n\n# tiny_program (S61): vocabulario autocontenido, estilo state_tracking.\n# Semantica: FACTS (FACT var val) + RULES (RULE rid lhs rhs) + query\n# (QUERY v0 r1..rd ANSWER). El modelo recibe el prefijo SIN el valor de\n# respuesta (prefix_answer=True: el target vive solo en y, nunca en x) y\n# debe encadenar reglas POR ID y responder con el FACT del var final.\n# depth = cantidad de rule-ids en la query; d2/d3 son buckets de la MISMA\n# task (bucket_fn), no tasks separadas (S61: sin doble peso en la barra).\nFACT_ID = 9\nRULE_ID = 10\nVOCAB_TINY_SIZE = 59\nN_VARS_TINY = 16\nN_VALS_TINY = 16\nN_RULES_TINY = 16\nVAR_OFFSET_TINY = 11\nVAL_OFFSET_TINY = 27\nRID_OFFSET_TINY = 43\n\n\ndef set_seed(seed: int):\n    random.seed(seed)\n    torch.manual_seed(seed)\n\n\ndef gen_mqar_sample(rng: random.Random, n_pairs: int = 3, hops: int = 1) -> List[int]:\n    if hops == 1:\n        keys = rng.sample(range(N_KEYS_MQAR), n_pairs)\n        vals = rng.sample(range(N_VALS_MQAR), n_pairs)\n        store = []\n        for k, v in zip(keys, vals):\n            store.append(K_OFFSET_MQAR + k)\n            store.append(V_OFFSET_MQAR + v)\n        idx = rng.randrange(n_pairs)\n        target_val = V_OFFSET_MQAR + vals[idx]\n        return ([BOS_ID] + store +\n                [SEP_ID, QUERY_ID, K_OFFSET_MQAR + keys[idx], ANSWER_ID, target_val])\n    # hops == 2: genuine 2-hop chain via value-as-key.\n    # Chain is always exactly 2 hops regardless of n_pairs:\n    #   query k_a=ids[0] -hop1-> v_a=ids[1] (acts as key k_b for hop 2)\n    #                                       -hop2-> v_b=ids[2]\n    # Store contains the two chain pairs (ids[0]->ids[1], ids[1]->ids[2]) PLUS\n    # (n_pairs-2) distractor pairs whose key/value are random ids NOT in the chain.\n    # Query gives only k_a; model must retrieve ids[1] from store, then look up\n    # ids[1] as a key in the store to find ids[2].\n    ids = rng.sample(range(max(N_KEYS_MQAR, N_VALS_MQAR)), 3)\n    chain_keys = [ids[0], ids[1]]\n    chain_vals = [ids[1], ids[2]]\n    k_query = chain_keys[0]\n    target_val = chain_vals[-1]\n    # Build distractor pairs from remaining ids not in the chain.\n    pool = [i for i in range(max(N_KEYS_MQAR, N_VALS_MQAR)) if i not in ids]\n    rng.shuffle(pool)\n    n_dist = max(0, n_pairs - 2)\n    distract_keys = pool[:n_dist]\n    distract_vals = pool[n_dist:2 * n_dist] if 2 * n_dist <= len(pool) else pool[:n_dist]\n    # Assemble pair list (chain + distractors), then shuffle pair order so the chain\n    # is not always at the front of the store.\n    pairs = list(zip(chain_keys, chain_vals)) + list(zip(distract_keys, distract_vals))\n    rng.shuffle(pairs)\n    store = []\n    for k, v in pairs:\n        store.append(K_OFFSET_MQAR + k)\n        store.append(V_OFFSET_MQAR + v)\n    # Query gives only k_a; the model must do hop 1 (retrieve v_a from store),\n    # then hop 2 (treat v_a as key, retrieve v_b from store).\n    return ([BOS_ID] + store +\n            [SEP_ID, QUERY_ID, K_OFFSET_MQAR + k_query, ANSWER_ID, V_OFFSET_MQAR + target_val])\n\n\ndef gen_forget_retrieve_sample(rng: random.Random, n_pairs: int = 6,\n                               n_forget: int = 1) -> List[int]:\n    # Quinta task (S45): store + instruccion FORGET k_i + QUERY k_j (j != i)\n    # + ANSWER v_j. Tests si el modelo descarta par borrado y aun recupera\n    # par presente. GDN/KDA/Qwen3-Next acumulan, no borran; s4d_carve tiene\n    # erase interno pero no ve instruccion FORGET explicita en tokens. Esta\n    # task debil comun deberia revelar donde el proximo mecanismo debe\n    # brillar (ver AGENTS.md "Las 5 sub-tasks").\n    #\n    # Formato:\n    #   BOS [k1 v1 ... kn vn] SEP FORGET k_i1 FORGET k_i2 ... SEP\n    #   QUERY k_j ANSWER v_j\n    #   donde j no esta en {i_1, ..., i_m} (recupera par NO borrado).\n    #\n    # Anti-atajos:\n    # - n_pairs >= 5 para que "v de cualquiera != v_i" no baste.\n    # - pares (k, v) con permutacion random sobre max(N_KEYS, N_VALS);\n    #   sin permutacion identity para evitar "v = k" como atajo.\n    # - n_forget <= n_pairs - 2 (siempre queda par presente a recuperar).\n    assert n_forget <= n_pairs - 2, \'need >=2 un-erased pairs for valid query\'\n    assert n_pairs >= 3, \'need >=3 pairs\'\n    ids_k = rng.sample(range(N_KEYS_MQAR), n_pairs)\n    ids_v = rng.sample(range(N_VALS_MQAR), n_pairs)\n    # Si un v_a == k_a por coincidencia de id, no es un atajo valido porque\n    # el modelo no puede saber "k -> v" hasta leer el store (no hay forma\n    # de derivar v de k por identidad cross-offset).\n    erased_idx = rng.sample(list(range(n_pairs)), n_forget)\n    erased_set = set(erased_idx)\n    remaining = [i for i in range(n_pairs) if i not in erased_set]\n    q_idx = rng.choice(remaining)\n    target_val = ids_v[q_idx]\n    # Construye store con orden de pares aleatorio (no revele erased vs alive).\n    pair_order = list(range(n_pairs))\n    rng.shuffle(pair_order)\n    store = []\n    for p in pair_order:\n        store.append(K_OFFSET_MQAR + ids_k[p])\n        store.append(V_OFFSET_MQAR + ids_v[p])\n    # Orden de FORGET aleatorio (si multiples erased).\n    forget_block = []\n    for ei in erased_idx:\n        forget_block.append(FORGET_ID)\n        forget_block.append(K_OFFSET_MQAR + ids_k[ei])\n    if len(forget_block) == 0:\n        # n_forget=0: degenerado; insertamos un marker neutro para no romper\n        # el parseo. (No usado en practica: n_forget>=1.)\n        forget_block = [FORGET_ID, K_OFFSET_MQAR + ids_k[0]]\n    # Forzar el q_idx NO erased ya garantizado por \'remaining\'.\n    return ([BOS_ID] + store +\n            [SEP_ID] + forget_block +\n            [SEP_ID, QUERY_ID, K_OFFSET_MQAR + ids_k[q_idx], ANSWER_ID,\n             V_OFFSET_MQAR + target_val])\n\n\nclass ForgetRetrieveDataset(Dataset):\n    def __init__(self, n_samples: int, seed: int = 0,\n                 n_pairs_range: Tuple[int, int] = (5, 9),\n                 n_forget_range: Tuple[int, int] = (1, 1)):\n        self.samples = []\n        rng = random.Random(seed)\n        seen = set()\n        attempts = 0\n        while len(self.samples) < n_samples and attempts < n_samples * 4:\n            attempts += 1\n            n_pairs = rng.randint(*n_pairs_range)\n            # n_forget en [1, n_pairs-2] para garantizar >=2 pares vivos.\n            max_f = max(1, n_pairs - 2)\n            lo_f, hi_f = n_forget_range\n            lo_f = max(1, min(lo_f, max_f))\n            hi_f = max(lo_f, min(hi_f, max_f))\n            n_forget = rng.randint(lo_f, hi_f)\n            seq = gen_forget_retrieve_sample(rng, n_pairs=n_pairs,\n                                             n_forget=n_forget)\n            key = tuple(seq)\n            if key in seen:\n                continue\n            seen.add(key)\n            self.samples.append(torch.tensor(seq, dtype=torch.long))\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getitem__(self, idx):\n        return self.samples[idx]\n\n\n# ---------------------------------------------------------------------------\n# tiny_program (S61): razonamiento composicional sobre reglas explicitas.\n# ---------------------------------------------------------------------------\n\ndef tiny_program_build_seq(rng, facts, rules, query_var, query_rules):\n    # facts: list[(var, val)], rules: list[(rid, lhs, rhs)]. Reordena\n    # facts y rules internamente (anti-atajo de orden) y construye el\n    # prefijo; termina en ANSWER_ID (el valor de respuesta NO va en x).\n    f = list(facts)\n    r = list(rules)\n    rng.shuffle(f)\n    rng.shuffle(r)\n    seq = [BOS_ID]\n    for var, val in f:\n        seq += [FACT_ID, VAR_OFFSET_TINY + var, VAL_OFFSET_TINY + val]\n    seq.append(SEP_ID)\n    for rid, lhs, rhs in r:\n        seq += [RULE_ID, RID_OFFSET_TINY + rid, VAR_OFFSET_TINY + lhs, VAR_OFFSET_TINY + rhs]\n    seq.append(SEP_ID)\n    seq += [QUERY_ID, VAR_OFFSET_TINY + query_var]\n    for rid in query_rules:\n        seq.append(RID_OFFSET_TINY + rid)\n    seq.append(ANSWER_ID)\n    return seq\n\n\ndef gen_tiny_program_sample(rng, depth=2, n_distract_facts=3, n_distract_rules=2,\n                            confusable_prob=0.4):\n    # Cadena: v0 --r1--> v1 --r2--> ... --rd--> vd. La query da v0 + ids de\n    # regla EN ORDEN; la respuesta es el FACT del var final. Anti-atajos:\n    # - vars/vals/ids re-sampleados por muestra (bindings frescos);\n    # - distractores: facts de vars fuera de la cadena y rules con ids no\n    #   consultados (cambiarlos no altera la respuesta);\n    # - confusables (prob confusable_prob): rule distractor con MISMO lhs\n    #   que una regla de la cadena y rhs distinto -> fuerza lookup por id.\n    chain_vars = rng.sample(range(N_VARS_TINY), depth + 1)\n    rule_ids = rng.sample(range(N_RULES_TINY), depth)\n    chain_vals = [rng.randrange(N_VALS_TINY) for _ in range(depth + 1)]\n    facts = list(zip(chain_vars, chain_vals))\n    rules = [(rule_ids[i], chain_vars[i], chain_vars[i + 1]) for i in range(depth)]\n    free_vars = [v for v in range(N_VARS_TINY) if v not in chain_vars]\n    rng.shuffle(free_vars)\n    for v in free_vars[:n_distract_facts]:\n        facts.append((v, rng.randrange(N_VALS_TINY)))\n    free_rids = [r for r in range(N_RULES_TINY) if r not in rule_ids]\n    rng.shuffle(free_rids)\n    for rid in free_rids[:n_distract_rules]:\n        if rng.random() < confusable_prob:\n            src = rng.choice(chain_vars[:-1])\n            nxt = chain_vars[chain_vars.index(src) + 1]\n            rhs = rng.choice([v for v in range(N_VARS_TINY) if v != nxt])\n            rules.append((rid, src, rhs))\n        else:\n            lhs = rng.choice(free_vars) if free_vars else chain_vars[0]\n            rhs = rng.choice([v for v in range(N_VARS_TINY) if v != lhs])\n            rules.append((rid, lhs, rhs))\n    return tiny_program_build_seq(rng, facts, rules, chain_vars[0], rule_ids)\n\n\ndef tiny_program_parse(x):\n    # Parsea el prefijo a estructura (dict) o None si malformada. Fuente\n    # unica de verdad para oracle y para las intervenciones del probe.\n    toks = [int(t) for t in x]\n    while toks and toks[-1] == PAD_ID:\n        toks.pop()\n    if not toks or toks[0] != BOS_ID:\n        return None\n    seps = [i for i, t in enumerate(toks) if t == SEP_ID]\n    if len(seps) < 2:\n        return None\n    f_end, r_end = seps[0], seps[1]\n    facts = {}\n    i = 1\n    while i < f_end:\n        if i + 2 >= f_end or toks[i] != FACT_ID:\n            return None\n        var = toks[i + 1] - VAR_OFFSET_TINY\n        val = toks[i + 2] - VAL_OFFSET_TINY\n        if not (0 <= var < N_VARS_TINY and 0 <= val < N_VALS_TINY):\n            return None\n        facts[var] = val\n        i += 3\n    rules = {}\n    i = f_end + 1\n    while i < r_end:\n        if i + 3 >= r_end or toks[i] != RULE_ID:\n            return None\n        rid = toks[i + 1] - RID_OFFSET_TINY\n        lhs = toks[i + 2] - VAR_OFFSET_TINY\n        rhs = toks[i + 3] - VAR_OFFSET_TINY\n        if not (0 <= rid < N_RULES_TINY and 0 <= lhs < N_VARS_TINY\n                and 0 <= rhs < N_VARS_TINY):\n            return None\n        rules[rid] = (lhs, rhs)\n        i += 4\n    q = toks[r_end + 1:]\n    if not q or q[-1] != ANSWER_ID or q[0] != QUERY_ID:\n        return None\n    q = q[1:-1]\n    if not q:\n        return None\n    q_var = q[0] - VAR_OFFSET_TINY\n    if not (0 <= q_var < N_VARS_TINY):\n        return None\n    q_rules = []\n    for t in q[1:]:\n        rid = t - RID_OFFSET_TINY\n        if not (0 <= rid < N_RULES_TINY):\n            return None\n        q_rules.append(rid)\n    return {\'facts\': facts, \'rules\': rules, \'query_var\': q_var, \'query_rules\': q_rules}\n\n\ndef tiny_program_oracle_target(x):\n    # Resuelve la respuesta del prefijo como parser de referencia: encadena\n    # reglas por id desde v0 y consulta el FACT del var final. None si la\n    # secuencia es invalida. Usado por tests y por el probe (targets de\n    # base e intervenciones).\n    p = tiny_program_parse(x)\n    if p is None:\n        return None\n    cur = p[\'query_var\']\n    for rid in p[\'query_rules\']:\n        r = p[\'rules\'].get(rid)\n        if r is None or r[0] != cur:\n            return None\n        cur = r[1]\n    val = p[\'facts\'].get(cur)\n    if val is None:\n        return None\n    return VAL_OFFSET_TINY + val\n\n\ndef tiny_depth_bucket(x):\n    # depth = cantidad de rule-ids en la query (d1/d2/d3; \'other\' si\n    # malformada). Separa EM_d2 (ID) de EM_d3 (OOD) dentro de la misma task.\n    p = tiny_program_parse(x)\n    if p is None:\n        return \'other\'\n    d = len(p[\'query_rules\'])\n    return \'d%d\' % d if d in (1, 2, 3) else \'other\'\n\n\ndef gen_dyck2_sample(rng, n_opened=3, n_closed_before=2, n_trailing=1):\n    """Prefijo Dyck-2 parcialmente balanceado con pila NO vacia al final.\n\n    La respuesta (oracle) es el CLOSE del tipo del TOP de la pila\n    (Suzgun et al. 2019, predict-closing-bracket).\n\n    Estructura del prefijo (stack machine):\n      1. n_closed_before pares balanceados de ruido (se cierran solos).\n      2. n_opened opens (quedan SIN cerrar; su profundidad define el bucket).\n      3. n_trailing pares balanceados interiores DESPUES de los opens:\n         sus closes sacan solo opens interiores; el top del stack final\n         sigue siendo opens[-1], un OPEN VIEJO.\n\n    Anticortos:\n    - \'copiar el ultimo token\': con n_trailing>=1 el ultimo token del\n      prefijo es un CLOSE interior (tipo distinto del top), nunca el par\n      del top.\n    - \'solo cuento la profundidad\': con n_opened>=2 hay que recordar el\n      tipo del mas interno abierto, no solo cuantos.\n    Formato: BOS [t1...tk] ANSWER (prefix_answer=True, el CLOSE va solo\n    en y; collate_fn de tiny_program ya maneja ese formato).\n    """\n    toks = []\n    for _ in range(n_closed_before):\n        t = rng.randrange(N_DYCK_TYPES)\n        toks.append(DYCK_OPEN_BASE + t)\n        toks.append(DYCK_CLOSE_BASE + t)\n    opens = [rng.randrange(N_DYCK_TYPES) for _ in range(n_opened)]\n    for t in opens:\n        toks.append(DYCK_OPEN_BASE + t)\n    for _ in range(n_trailing):\n        t = rng.randrange(N_DYCK_TYPES)\n        toks.append(DYCK_OPEN_BASE + t)\n        toks.append(DYCK_CLOSE_BASE + t)\n    seq = [BOS_ID] + toks + [ANSWER_ID]\n    target = DYCK_CLOSE_BASE + opens[-1]\n    return seq, target\n\n\ndef dyck2_oracle_target(seq):\n    """CLOSE del top de la pila del prefijo, o None si malformada."""\n    toks = [int(t) for t in seq]\n    while toks and toks[-1] == PAD_ID:\n        toks.pop()\n    if not toks or toks[0] != BOS_ID or toks[-1] != ANSWER_ID:\n        return None\n    stack = []\n    for t in toks[1:-1]:\n        if DYCK_OPEN_BASE <= t < DYCK_OPEN_BASE + N_DYCK_TYPES:\n            stack.append(t - DYCK_OPEN_BASE)\n        elif DYCK_CLOSE_BASE <= t < DYCK_CLOSE_BASE + N_DYCK_TYPES:\n            if not stack:\n                return None\n            stack.pop()\n        else:\n            return None\n    if not stack:\n        return None\n    return DYCK_CLOSE_BASE + stack[-1]\n\n\ndef dyck2_depth_bucket(x):\n    """Bucket por profundidad de la pila al final del prefijo (d1/d2/d3+)."""\n    p = dyck2_oracle_target(x)\n    if p is None:\n        return \'other\'\n    toks = [int(t) for t in x]\n    while toks and toks[-1] == PAD_ID:\n        toks.pop()\n    depth = 0\n    for t in toks[1:-1]:\n        if DYCK_OPEN_BASE <= t < DYCK_OPEN_BASE + N_DYCK_TYPES:\n            depth += 1\n        elif DYCK_CLOSE_BASE <= t < DYCK_CLOSE_BASE + N_DYCK_TYPES:\n            depth -= 1\n    if depth <= 2:\n        return \'d%d\' % depth\n    return \'d3+\'\n\n\nclass Dyck2Dataset(Dataset):\n    # Tuplas (seq_prefijo, target): el CLOSE respuesta NO esta en x\n    # (TaskSpec usa prefix_answer=True, como tiny_program).\n    def __init__(self, n_samples, seed=0, n_opened_range=(2, 4),\n                 n_closed_range=(1, 4), n_trailing_range=(1, 3)):\n        self.samples = []\n        rng = random.Random(seed)\n        seen = set()\n        attempts = 0\n        while len(self.samples) < n_samples and attempts < n_samples * 8:\n            attempts += 1\n            n_opened = rng.randint(*n_opened_range)\n            n_closed = rng.randint(*n_closed_range)\n            n_trail = rng.randint(*n_trailing_range)\n            seq, tgt = gen_dyck2_sample(rng, n_opened=n_opened,\n                                        n_closed_before=n_closed,\n                                        n_trailing=n_trail)\n            if dyck2_oracle_target(seq) != tgt:\n                continue\n            key = tuple(seq)\n            if key in seen:\n                continue\n            seen.add(key)\n            self.samples.append((torch.tensor(seq, dtype=torch.long),\n                                 torch.tensor(tgt, dtype=torch.long)))\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getitem__(self, idx):\n        return self.samples[idx]\n\n\nclass TinyProgramDataset(Dataset):\n    # Entrega tuplas (seq_prefijo, target): el valor de respuesta NO esta\n    # en x (el TaskSpec de la task usa prefix_answer=True; collate_fn lo\n    # pone en y en la posicion del ANSWER_ID).\n    def __init__(self, n_samples, seed=0, depth_range=(1, 2),\n                 n_facts_range=(2, 4), n_rules_range=(2, 3),\n                 confusable_prob=0.4):\n        self.samples = []\n        rng = random.Random(seed)\n        seen = set()\n        attempts = 0\n        while len(self.samples) < n_samples and attempts < n_samples * 8:\n            attempts += 1\n            depth = rng.randint(*depth_range)\n            seq = gen_tiny_program_sample(\n                rng, depth=depth,\n                n_distract_facts=rng.randint(*n_facts_range),\n                n_distract_rules=rng.randint(*n_rules_range),\n                confusable_prob=confusable_prob)\n            tgt = tiny_program_oracle_target(seq)\n            if tgt is None:\n                continue\n            key = tuple(seq)\n            if key in seen:\n                continue\n            seen.add(key)\n            self.samples.append((torch.tensor(seq, dtype=torch.long),\n                                 torch.tensor(tgt, dtype=torch.long)))\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getitem__(self, idx):\n        return self.samples[idx]\n\n\nclass MQARDataset(Dataset):\n    def __init__(self, n_samples: int, n_pairs_range: Tuple[int, int] = (2, 4),\n                 hops: int = 1, seed: int = 0):\n        self.samples = []\n        rng = random.Random(seed)\n        seen = set()\n        attempts = 0\n        while len(self.samples) < n_samples and attempts < n_samples * 4:\n            attempts += 1\n            n_pairs = rng.randint(*n_pairs_range)\n            seq = gen_mqar_sample(rng, n_pairs=n_pairs, hops=hops)\n            key = tuple(seq)\n            if key in seen:\n                continue\n            seen.add(key)\n            self.samples.append(torch.tensor(seq, dtype=torch.long))\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getitem__(self, idx):\n        return self.samples[idx]\n\n\ndef gen_copy_reverse_sample(rng: random.Random, min_len: int = 8, max_len: int = 12, shift_aug_max: int = 0) -> List[int]:\n    n = rng.randint(min_len, max_len)\n    digits = [DIGIT_OFFSET + rng.randrange(N_DIGITS_COPY) for _ in range(n)]\n    rev = list(reversed(digits))\n    seq = [BOS_ID] + digits + [SEP_ID, ANSWER_ID] + rev + [EOS_ID]\n    if shift_aug_max > 0:\n        offset = rng.randint(0, shift_aug_max)\n        seq = [PAD_ID] * offset + seq\n    return seq\n\n\nclass CopyReverseDataset(Dataset):\n    def __init__(self, n_samples: int, seed: int = 0,\n                 min_len: int = 8, max_len: int = 12,\n                 shift_aug_max: int = 0):\n        self.samples = []\n        rng = random.Random(seed)\n        seen = set()\n        attempts = 0\n        while len(self.samples) < n_samples and attempts < n_samples * 4:\n            attempts += 1\n            seq = gen_copy_reverse_sample(rng, min_len, max_len, shift_aug_max=shift_aug_max)\n            key = tuple(seq)\n            if key in seen:\n                continue\n            seen.add(key)\n            self.samples.append(torch.tensor(seq, dtype=torch.long))\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getitem__(self, idx):\n        return self.samples[idx]\n\n\ndef gen_state_tracking_sample(rng: random.Random, n_events: int = 8) -> List[int]:\n    ent_loc = {e: rng.randrange(N_LOCATIONS) for e in range(N_ENTITIES)}\n    seq = [BOS_ID]\n    for _ in range(n_events):\n        et = rng.randint(0, N_ENTITIES - 1)\n        op = rng.choice([\'assign\', \'goto\'])\n        loc = rng.randrange(N_LOCATIONS)\n        ent_loc[et] = loc\n        op_id = ASSIGN_ID if op == \'assign\' else GOTO_ID\n        seq += [ENT_OFFSET + et, op_id, LOC_OFFSET + loc, SEP_ID]\n    qe = rng.randrange(N_ENTITIES)\n    target_loc = ent_loc[qe]\n    seq += [QUERY_ID, ENT_OFFSET + qe, ANSWER_ID, LOC_OFFSET + target_loc]\n    return seq\n\n\nclass StateTrackingDataset(Dataset):\n    def __init__(self, n_samples: int, seed: int = 0,\n                 n_events_range: Tuple[int, int] = (6, 10)):\n        self.samples = []\n        rng = random.Random(seed)\n        seen = set()\n        attempts = 0\n        while len(self.samples) < n_samples and attempts < n_samples * 4:\n            attempts += 1\n            n_events = rng.randint(*n_events_range)\n            seq = gen_state_tracking_sample(rng, n_events=n_events)\n            key = tuple(seq)\n            if key in seen:\n                continue\n            seen.add(key)\n            self.samples.append(torch.tensor(seq, dtype=torch.long))\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getitem__(self, idx):\n        return self.samples[idx]\n\n\ndef make_target_shifted(input_ids: torch.Tensor, pad_id: int = PAD_ID,\n                        answer_marker_id: Optional[int] = None,\n                        mark_after_marker: bool = False):\n    B, T = input_ids.shape\n    y = torch.full_like(input_ids, pad_id)\n    target_mask = torch.zeros_like(input_ids, dtype=torch.bool)\n    y[:, :-1] = input_ids[:, 1:]\n    if answer_marker_id is None:\n        target_mask[:, :-1] = input_ids[:, 1:] != pad_id\n    elif not mark_after_marker:\n        target_mask[:, :-1] = input_ids[:, :-1] == answer_marker_id\n    else:\n        marker_pos = (input_ids == answer_marker_id).int().argmax(dim=1)\n        has_marker = (input_ids == answer_marker_id).any(dim=1)\n        last_nonpad = (input_ids != pad_id).int().sum(dim=1) - 1\n        for b in range(B):\n            if not bool(has_marker[b]):\n                continue\n            s = int(marker_pos[b])\n            e = int(last_nonpad[b].clamp(min=s))\n            target_mask[b, s:e+1] = True\n            if y[b, e] == pad_id:\n                target_mask[b, e] = False\n    return y, target_mask\n\n\ndef collate_fn(batch: List[torch.Tensor], pad_id: int = PAD_ID,\n               answer_marker_id: Optional[int] = None, mark_after_marker: bool = False,\n               prefix_answer: bool = False):\n    # prefix_answer (S61): los items del batch son tuplas (seq, target);\n    # seq termina en ANSWER (ultimo token real) y el target (el valor de\n    # respuesta) vive SOLO en y, nunca en x. Aplica a train Y eval por\n    # construccion: el modelo nunca ve tokens futuros dentro del chunk.\n    if prefix_answer:\n        assert answer_marker_id is not None, \'prefix_answer requires answer_marker_id\'\n        seqs = [b[0] for b in batch]\n        targets = [int(b[1]) for b in batch]\n        lengths = [len(s) for s in seqs]\n        T = max(lengths)\n        x = torch.full((len(batch), T), pad_id, dtype=torch.long)\n        for i, s in enumerate(seqs):\n            x[i, :len(s)] = s\n        y = torch.full_like(x, pad_id)\n        target_mask = torch.zeros_like(x, dtype=torch.bool)\n        for i in range(len(batch)):\n            pos = (x[i] == answer_marker_id).nonzero(as_tuple=False)\n            if len(pos) == 0:\n                continue\n            p = int(pos[0].item())\n            y[i, p] = targets[i]\n            target_mask[i, p] = True\n        return x, y, target_mask\n    lengths = [len(s) for s in batch]\n    T = max(lengths)\n    x = torch.full((len(batch), T), pad_id, dtype=torch.long)\n    for i, s in enumerate(batch):\n        x[i, :len(s)] = s\n    y, target_mask = make_target_shifted(x, pad_id=pad_id,\n                                          answer_marker_id=answer_marker_id,\n                                          mark_after_marker=mark_after_marker)\n    return x, y, target_mask\n\n\n@dataclass\nclass TaskSpec:\n    name: str\n    vocab_size: int\n    max_seq_len: int\n    epochs: int\n    train_samples: int\n    valid_samples: int\n    batch_size: int\n    success_threshold: float\n    success_metric: str\n    make_train: Callable\n    make_valid: Callable\n    answer_marker_id: Optional[int] = None\n    mark_after_marker: bool = False\n    # S61 (tiny_program): el dataset entrega tuplas (seq, target) y el\n    # target vive solo en y (nunca en x). bucket_fn etiqueta cada muestra\n    # (ej. d2/d3) para reportar EM separado por bucket dentro de la task;\n    # bucket_thresholds gatea recommend_kaggle: cada bucket debe pasar su\n    # umbral (el gate real de extrapolacion es d3).\n    prefix_answer: bool = False\n    bucket_fn: Optional[Callable] = None\n    bucket_thresholds: Optional[Dict[str, float]] = None\n\n\ndef make_task_specs(smoke: bool = True, include_forget: bool = True,\n                    include_tiny_program: bool = False,\n                    include_dyck2: bool = False) -> List[TaskSpec]:\n    # include_forget=True desde 2026-08-03 (S45): la 5a task\n    # `forget_retrieval` (store + FORGET k_i + QUERY k_j != i + ANSWER v_j)\n    # entra al harness por defecto. Modo legacy 4-tasks:\n    # `make_task_specs(smoke=..., include_forget=False)`. La regla\n    # recommend_kaggle (passed >= 3 AND passes_baseline >= 2) reescala a\n    # `passed >= n_total - 2` con n_total=5 -> >=4 (ver `_recommend_kaggle`).\n    # include_tiny_program (S61): task 6ta `tiny_program` OP-TIN (default\n    # False) para calibrar sin invalidar el baseline historico. El runner\n    # la activa via `--tasks tiny_program` (run_smoke lo detecta) o con el\n    # flag del runner. d2/d3 son buckets de la misma task (EM separado via\n    # bucket_fn), con d3 como gate de extrapolacion (bucket_thresholds).\n    # include_dyck2 (S81): task 7ma `dyck2` OP-TIN (default False), misma\n    # mecanica que tiny_program. Mide memoria de stack / composicion\n    # (Dyck-2 predict-closing-bracket); bucket d3+ gatea la profundidad.\n    tiny = None\n    dyck2 = None\n    if smoke == \'cpu_quick\':\n        specs = [\n            TaskSpec(\n                name=\'mqar_1hop\',\n                vocab_size=VOCAB_MQAR_SIZE, max_seq_len=24, epochs=60,\n                train_samples=400, valid_samples=120, batch_size=32,\n                success_threshold=0.85, success_metric=\'exact_match\',\n                make_train=lambda seed: MQARDataset(400, seed=seed, hops=1, n_pairs_range=(5, 9)),\n                make_valid=lambda seed: MQARDataset(120, seed=seed + 1, hops=1, n_pairs_range=(5, 9)),\n                answer_marker_id=ANSWER_ID,\n            ),\n            TaskSpec(\n                name=\'mqar_2hop\',\n                vocab_size=VOCAB_MQAR_SIZE, max_seq_len=24, epochs=60,\n                train_samples=400, valid_samples=120, batch_size=32,\n                success_threshold=0.80, success_metric=\'exact_match\',\n                make_train=lambda seed: MQARDataset(400, seed=seed, hops=2, n_pairs_range=(5, 9)),\n                make_valid=lambda seed: MQARDataset(120, seed=seed + 1, hops=2, n_pairs_range=(5, 9)),\n                answer_marker_id=ANSWER_ID,\n            ),\n            TaskSpec(\n                name=\'copy_reverse\',\n                vocab_size=VOCAB_COPY_SIZE, max_seq_len=48, epochs=80,\n                train_samples=800, valid_samples=200, batch_size=32,\n                success_threshold=0.90, success_metric=\'exact_match\',\n                make_train=lambda seed: CopyReverseDataset(800, seed=seed, min_len=20, max_len=22),\n                make_valid=lambda seed: CopyReverseDataset(200, seed=seed + 1, min_len=20, max_len=22),\n                answer_marker_id=ANSWER_ID, mark_after_marker=True,\n            ),\n            TaskSpec(\n                name=\'state_tracking\',\n                vocab_size=VOCAB_ST_SIZE, max_seq_len=80, epochs=80,\n                train_samples=600, valid_samples=150, batch_size=32,\n                success_threshold=0.70, success_metric=\'exact_match\',\n                make_train=lambda seed: StateTrackingDataset(600, seed=seed, n_events_range=(6, 10)),\n                make_valid=lambda seed: StateTrackingDataset(150, seed=seed + 1, n_events_range=(6, 10)),\n                answer_marker_id=ANSWER_ID,\n            ),\n            TaskSpec(\n                name=\'forget_retrieval\',\n                vocab_size=VOCAB_FORGET_SIZE, max_seq_len=32, epochs=80,\n                train_samples=600, valid_samples=150, batch_size=32,\n                success_threshold=0.70, success_metric=\'exact_match\',\n                make_train=lambda seed: ForgetRetrieveDataset(600, seed=seed, n_pairs_range=(5, 9), n_forget_range=(1, 2)),\n                make_valid=lambda seed: ForgetRetrieveDataset(150, seed=seed + 1, n_pairs_range=(5, 9), n_forget_range=(1, 2)),\n                answer_marker_id=ANSWER_ID,\n            ),\n        ]\n        tiny = TaskSpec(\n            name=\'tiny_program\',\n            vocab_size=VOCAB_TINY_SIZE, max_seq_len=64, epochs=80,\n            train_samples=600, valid_samples=200, batch_size=32,\n            success_threshold=0.60, success_metric=\'exact_match\',\n            make_train=lambda seed: TinyProgramDataset(600, seed=seed, depth_range=(1, 2)),\n            make_valid=lambda seed: TinyProgramDataset(200, seed=seed + 1, depth_range=(2, 3)),\n            answer_marker_id=ANSWER_ID, prefix_answer=True,\n            bucket_fn=tiny_depth_bucket,\n            bucket_thresholds={\'d3\': 0.40},\n        )\n        dyck2 = TaskSpec(\n            name=\'dyck2\',\n            vocab_size=VOCAB_DYCK_SIZE, max_seq_len=24, epochs=80,\n            train_samples=600, valid_samples=200, batch_size=32,\n            success_threshold=0.70, success_metric=\'exact_match\',\n            make_train=lambda seed: Dyck2Dataset(600, seed=seed),\n            make_valid=lambda seed: Dyck2Dataset(200, seed=seed + 1),\n            answer_marker_id=ANSWER_ID, prefix_answer=True,\n            bucket_fn=dyck2_depth_bucket,\n            bucket_thresholds={\'d3+\': 0.40},\n        )\n    elif smoke == \'smoke\' or smoke is True:\n        specs = [\n            TaskSpec(\n                name=\'mqar_1hop\',\n                vocab_size=VOCAB_MQAR_SIZE, max_seq_len=42, epochs=200,\n                train_samples=800, valid_samples=200, batch_size=16,\n                success_threshold=0.85, success_metric=\'exact_match\',\n                make_train=lambda seed: MQARDataset(800, seed=seed, hops=1, n_pairs_range=(12, 18)),\n                make_valid=lambda seed: MQARDataset(200, seed=seed + 1, hops=1, n_pairs_range=(12, 18)),\n                answer_marker_id=ANSWER_ID,\n            ),\n            TaskSpec(\n                name=\'mqar_2hop\',\n                vocab_size=VOCAB_MQAR_SIZE, max_seq_len=42, epochs=250,\n                train_samples=800, valid_samples=200, batch_size=16,\n                success_threshold=0.80, success_metric=\'exact_match\',\n                make_train=lambda seed: MQARDataset(800, seed=seed, hops=2, n_pairs_range=(12, 18)),\n                make_valid=lambda seed: MQARDataset(200, seed=seed + 1, hops=2, n_pairs_range=(12, 18)),\n                answer_marker_id=ANSWER_ID,\n            ),\n            TaskSpec(\n                name=\'copy_reverse\',\n                vocab_size=VOCAB_COPY_SIZE, max_seq_len=48, epochs=300,\n                train_samples=1600, valid_samples=400, batch_size=16,\n                success_threshold=0.90, success_metric=\'exact_match\',\n                make_train=lambda seed: CopyReverseDataset(1600, seed=seed, min_len=20, max_len=22),\n                make_valid=lambda seed: CopyReverseDataset(400, seed=seed + 1, min_len=20, max_len=22),\n                answer_marker_id=ANSWER_ID, mark_after_marker=True,\n            ),\n            TaskSpec(\n                name=\'state_tracking\',\n                vocab_size=VOCAB_ST_SIZE, max_seq_len=80, epochs=400,\n                train_samples=1200, valid_samples=300, batch_size=16,\n                success_threshold=0.70, success_metric=\'exact_match\',\n                make_train=lambda seed: StateTrackingDataset(1200, seed=seed, n_events_range=(6, 10)),\n                make_valid=lambda seed: StateTrackingDataset(300, seed=seed + 1, n_events_range=(6, 10)),\n                answer_marker_id=ANSWER_ID,\n            ),\n            TaskSpec(\n                name=\'forget_retrieval\',\n                vocab_size=VOCAB_FORGET_SIZE, max_seq_len=48, epochs=300,\n                train_samples=1200, valid_samples=300, batch_size=16,\n                success_threshold=0.70, success_metric=\'exact_match\',\n                make_train=lambda seed: ForgetRetrieveDataset(1200, seed=seed, n_pairs_range=(12, 18), n_forget_range=(1, 3)),\n                make_valid=lambda seed: ForgetRetrieveDataset(300, seed=seed + 1, n_pairs_range=(12, 18), n_forget_range=(1, 3)),\n                answer_marker_id=ANSWER_ID,\n            ),\n        ]\n        tiny = TaskSpec(\n            name=\'tiny_program\',\n            vocab_size=VOCAB_TINY_SIZE, max_seq_len=80, epochs=300,\n            train_samples=1200, valid_samples=400, batch_size=16,\n            success_threshold=0.60, success_metric=\'exact_match\',\n            make_train=lambda seed: TinyProgramDataset(1200, seed=seed, depth_range=(1, 2), n_facts_range=(4, 7), n_rules_range=(3, 6)),\n            make_valid=lambda seed: TinyProgramDataset(400, seed=seed + 1, depth_range=(2, 3), n_facts_range=(4, 7), n_rules_range=(3, 6)),\n            answer_marker_id=ANSWER_ID, prefix_answer=True,\n            bucket_fn=tiny_depth_bucket,\n            bucket_thresholds={\'d3\': 0.40},\n        )\n        dyck2 = TaskSpec(\n            name=\'dyck2\',\n            vocab_size=VOCAB_DYCK_SIZE, max_seq_len=48, epochs=300,\n            train_samples=1200, valid_samples=400, batch_size=16,\n            success_threshold=0.70, success_metric=\'exact_match\',\n            make_train=lambda seed: Dyck2Dataset(1200, seed=seed, n_opened_range=(2, 5), n_closed_range=(2, 6), n_trailing_range=(2, 6)),\n            make_valid=lambda seed: Dyck2Dataset(400, seed=seed + 1, n_opened_range=(2, 5), n_closed_range=(2, 6), n_trailing_range=(2, 6)),\n            answer_marker_id=ANSWER_ID, prefix_answer=True,\n            bucket_fn=dyck2_depth_bucket,\n            bucket_thresholds={\'d3+\': 0.40},\n        )\n    else:\n        specs = [\n            TaskSpec(\n                name=\'mqar_1hop\', vocab_size=VOCAB_MQAR_SIZE, max_seq_len=54, epochs=400,\n                train_samples=1000, valid_samples=200, batch_size=32,\n                success_threshold=0.85, success_metric=\'exact_match\',\n                make_train=lambda seed: MQARDataset(1000, seed=seed, hops=1, n_pairs_range=(16, 24)),\n                make_valid=lambda seed: MQARDataset(200, seed=seed + 1, hops=1, n_pairs_range=(16, 24)),\n                answer_marker_id=ANSWER_ID,\n            ),\n            TaskSpec(\n                name=\'mqar_2hop\', vocab_size=VOCAB_MQAR_SIZE, max_seq_len=54, epochs=500,\n                train_samples=1000, valid_samples=200, batch_size=32,\n                success_threshold=0.80, success_metric=\'exact_match\',\n                make_train=lambda seed: MQARDataset(1000, seed=seed, hops=2, n_pairs_range=(16, 24)),\n                make_valid=lambda seed: MQARDataset(200, seed=seed + 1, hops=2, n_pairs_range=(16, 24)),\n                answer_marker_id=ANSWER_ID,\n            ),\n            TaskSpec(\n                name=\'copy_reverse\', vocab_size=VOCAB_COPY_SIZE, max_seq_len=48, epochs=500,\n                train_samples=2000, valid_samples=400, batch_size=32,\n                success_threshold=0.90, success_metric=\'exact_match\',\n                make_train=lambda seed: CopyReverseDataset(2000, seed=seed, min_len=20, max_len=22),\n                make_valid=lambda seed: CopyReverseDataset(400, seed=seed + 1, min_len=20, max_len=22),\n                answer_marker_id=ANSWER_ID, mark_after_marker=True,\n            ),\n            TaskSpec(\n                name=\'state_tracking\', vocab_size=VOCAB_ST_SIZE, max_seq_len=80, epochs=600,\n                train_samples=1500, valid_samples=300, batch_size=32,\n                success_threshold=0.70, success_metric=\'exact_match\',\n                make_train=lambda seed: StateTrackingDataset(1500, seed=seed, n_events_range=(6, 10)),\n                make_valid=lambda seed: StateTrackingDataset(300, seed=seed + 1, n_events_range=(6, 10)),\n                answer_marker_id=ANSWER_ID,\n            ),\n            TaskSpec(\n                name=\'forget_retrieval\', vocab_size=VOCAB_FORGET_SIZE, max_seq_len=54, epochs=600,\n                train_samples=1500, valid_samples=300, batch_size=32,\n                success_threshold=0.70, success_metric=\'exact_match\',\n                make_train=lambda seed: ForgetRetrieveDataset(1500, seed=seed, n_pairs_range=(16, 24), n_forget_range=(1, 4)),\n                make_valid=lambda seed: ForgetRetrieveDataset(300, seed=seed + 1, n_pairs_range=(16, 24), n_forget_range=(1, 4)),\n                answer_marker_id=ANSWER_ID,\n            ),\n        ]\n        tiny = TaskSpec(\n            name=\'tiny_program\',\n            vocab_size=VOCAB_TINY_SIZE, max_seq_len=96, epochs=600,\n            train_samples=1500, valid_samples=500, batch_size=32,\n            success_threshold=0.60, success_metric=\'exact_match\',\n            make_train=lambda seed: TinyProgramDataset(1500, seed=seed, depth_range=(1, 2), n_facts_range=(6, 10), n_rules_range=(4, 8)),\n            make_valid=lambda seed: TinyProgramDataset(500, seed=seed + 1, depth_range=(2, 3), n_facts_range=(6, 10), n_rules_range=(4, 8)),\n            answer_marker_id=ANSWER_ID, prefix_answer=True,\n            bucket_fn=tiny_depth_bucket,\n            bucket_thresholds={\'d3\': 0.40},\n        )\n        dyck2 = TaskSpec(\n            name=\'dyck2\',\n            vocab_size=VOCAB_DYCK_SIZE, max_seq_len=64, epochs=500,\n            train_samples=1500, valid_samples=500, batch_size=32,\n            success_threshold=0.70, success_metric=\'exact_match\',\n            make_train=lambda seed: Dyck2Dataset(1500, seed=seed, n_opened_range=(2, 6), n_closed_range=(3, 8), n_trailing_range=(3, 8)),\n            make_valid=lambda seed: Dyck2Dataset(500, seed=seed + 1, n_opened_range=(2, 6), n_closed_range=(3, 8), n_trailing_range=(3, 8)),\n            answer_marker_id=ANSWER_ID, prefix_answer=True,\n            bucket_fn=dyck2_depth_bucket,\n            bucket_thresholds={\'d3+\': 0.40},\n        )\n    if include_tiny_program and tiny is not None:\n        specs.append(tiny)\n    if include_dyck2 and dyck2 is not None:\n        specs.append(dyck2)\n    if not include_forget:\n        specs = [s for s in specs if s.name != \'forget_retrieval\']\n    return specs\n\n\ndef train_model(model, train_loader, valid_loader, device, epochs: int,\n                lr: float, weight_decay: float, clip: float, vocab_size: int,\n                optimizer: str = \'adamw\', loss_fn: Optional[Callable] = None,\n                bucket_fn: Optional[Callable] = None,\n                ss_fn: Optional[Callable] = None):\n    # loss_fn (opcional, default None): fn(model, logits, y_train, vocab_size)\n    # -> scalar. Se usa para anadir terminos auxiliares (ej. L_homeo del\n    # s4d_valence). None = CE puro (comportamiento historico identico).\n    # bucket_fn (S61): etiqueta cada muestra (ej. tiny d2/d3) -> trackea\n    # EM por bucket + max-in-window por bucket (precedente S30.11).\n    from tests.muon_opt import make_optimizer\n    model.to(device)\n    opt = make_optimizer(model, optimizer, lr=lr, weight_decay=weight_decay)\n    hist = []\n    max_em_in_window = 0.0  # S30.11: blips transitorios -> reporta max\n    max_token_acc_window = 0.0\n    bucket_max = {}\n    t0 = time.time()\n    for ep in range(1, epochs + 1):\n        model.train()\n        ep_t0 = time.time()\n        train_loss, train_tokens = 0.0, 0\n        for x, y, target_mask in train_loader:\n            x = x.to(device)\n            y_train = torch.where(target_mask, y, torch.full_like(y, PAD_ID))\n            y_train = y_train.to(device)\n            opt.zero_grad(set_to_none=True)\n            if ss_fn is not None:\n                x = ss_fn(x, target_mask, model, ep)\n            logits = model(x)\n            if loss_fn is not None:\n                loss = loss_fn(model, logits, y_train, vocab_size)\n            else:\n                loss = F.cross_entropy(logits.reshape(-1, vocab_size), y_train.reshape(-1),\n                                       ignore_index=PAD_ID)\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), clip)\n            opt.step()\n            n_tok = (y_train != PAD_ID).sum().item()\n            train_loss += loss.item() * n_tok\n            train_tokens += n_tok\n        val = evaluate(model, valid_loader, device, vocab_size, bucket_fn=bucket_fn)\n        max_em_in_window = max(max_em_in_window, val[\'exact_match\'])\n        max_token_acc_window = max(max_token_acc_window, val[\'token_acc\'])\n        for k, v in val.get(\'exact_match_by_bucket\', {}).items():\n            bucket_max[k] = max(bucket_max.get(k, 0.0), v)\n        hist.append({\n            \'epoch\': ep,\n            \'train_loss\': train_loss / max(train_tokens, 1),\n            \'valid_loss\': val[\'loss\'],\n            \'valid_token_acc\': val[\'token_acc\'],\n            \'valid_exact_match\': val[\'exact_match\'],\n            \'valid_max_em_in_window\': max_em_in_window,\n            \'valid_max_token_acc_window\': max_token_acc_window,\n            \'valid_exact_match_by_bucket\': val.get(\'exact_match_by_bucket\', {}),\n            \'epoch_seconds\': time.time() - ep_t0,\n        })\n    final = hist[-1] if hist else {\'valid_loss\': float(\'nan\'),\n                                    \'valid_token_acc\': 0.0,\n                                    \'valid_exact_match\': 0.0}\n    final_meta = {\n        \'total_seconds\': time.time() - t0,\n        \'final\': final,\n        \'max_em_in_window\': max_em_in_window,\n        \'max_token_acc_window\': max_token_acc_window,\n        \'max_em_by_bucket_in_window\': bucket_max,\n    }\n    return hist, final_meta\n\n\ndef evaluate(model, loader, device, vocab_size: int,\n             bucket_fn: Optional[Callable] = None) -> Dict[str, float]:\n    model.eval()\n    total_loss, total_tokens, total_correct = 0.0, 0, 0\n    exact_match_ok, exact_match_n = 0, 0\n    bucket_ok, bucket_n = {}, {}\n    with torch.no_grad():\n        for x, y, target_mask in loader:\n            x = x.to(device)\n            y_eval = torch.where(target_mask, y, torch.full_like(y, PAD_ID))\n            y_eval = y_eval.to(device)\n            logits = model(x)\n            loss = F.cross_entropy(logits.reshape(-1, vocab_size), y_eval.reshape(-1),\n                                   ignore_index=PAD_ID, reduction=\'sum\')\n            n_tok = (y_eval != PAD_ID).sum().item()\n            total_loss += loss.item()\n            total_tokens += n_tok\n            pred = logits.argmax(dim=-1)\n            correct = (pred == y_eval) & (y_eval != PAD_ID)\n            total_correct += correct.sum().item()\n            for i in range(x.size(0)):\n                m = target_mask[i]\n                if m.sum().item() == 0:\n                    continue\n                exact_match_n += 1\n                pos = torch.nonzero(m, as_tuple=False).squeeze(-1)\n                matched = torch.equal(pred[i][pos], y_eval[i][pos])\n                if matched:\n                    exact_match_ok += 1\n                if bucket_fn is not None:\n                    lab = bucket_fn(x[i])\n                    bucket_n[lab] = bucket_n.get(lab, 0) + 1\n                    if matched:\n                        bucket_ok[lab] = bucket_ok.get(lab, 0) + 1\n    res = {\n        \'loss\': total_loss / max(total_tokens, 1),\n        \'token_acc\': total_correct / max(total_tokens, 1),\n        \'exact_match\': exact_match_ok / max(exact_match_n, 1),\n    }\n    if bucket_fn is not None:\n        res[\'exact_match_by_bucket\'] = {\n            k: bucket_ok.get(k, 0) / max(bucket_n.get(k, 0), 1)\n            for k in bucket_n\n        }\n    return res\n\n\ndef run_one_task(task: TaskSpec, build_fn: Callable, device, seed: int,\n                 lr: float, weight_decay: float, clip: float,\n                 d_model: int, n_layers: int, variant: str = \'unknown\',\n                 compile_model: bool = False,\n                 optimizer: str = \'adamw\',\n                 loss_fn: Optional[Callable] = None,\n                 ss_fn: Optional[Callable] = None) -> Dict:\n    from functools import partial\n    if device.type == \'cuda\':\n        # Tecnicas GPU de bajo riesgo (DELIBERATION 30.15): TF32 en\n        # matmuls+convs (T4) y cudnn.benchmark. El gate CPU local queda\n        # en fp32 exacto (no afecta). bf16/autocast queda fuera (cambio\n        # numerico grande -> re-validacion completa).\n        torch.set_float32_matmul_precision(\'high\')\n        torch.backends.cudnn.benchmark = True\n    set_seed(seed)\n    train_ds = task.make_train(seed=seed)\n    valid_ds = task.make_valid(seed=seed + 100)\n    col = partial(collate_fn, answer_marker_id=task.answer_marker_id,\n                  mark_after_marker=task.mark_after_marker,\n                  prefix_answer=task.prefix_answer)\n    train_loader = DataLoader(train_ds, batch_size=task.batch_size, shuffle=True,\n                              collate_fn=col, num_workers=0)\n    valid_loader = DataLoader(valid_ds, batch_size=task.batch_size, shuffle=False,\n                             collate_fn=col, num_workers=0)\n    model = build_fn(vocab_size=task.vocab_size, max_len=task.max_seq_len,\n                     d_model=d_model, n_layers=n_layers)\n    if compile_model:\n        model = torch.compile(model)\n    n_params = sum(p.numel() for p in model.parameters())\n    t0 = time.time()\n    hist, meta = train_model(model, train_loader, valid_loader, device,\n                             epochs=task.epochs, lr=lr, weight_decay=weight_decay,\n                             clip=clip, vocab_size=task.vocab_size,\n                             optimizer=optimizer, loss_fn=loss_fn,\n                             bucket_fn=task.bucket_fn, ss_fn=ss_fn)\n    final = meta[\'final\']\n    metric = final[\'valid_\' + task.success_metric]\n    success = metric >= task.success_threshold\n    ret = {\n        \'task\': task.name,\n        \'n_params\': n_params,\n        \'epochs\': task.epochs,\n        \'train_samples\': task.train_samples,\n        \'valid_samples\': task.valid_samples,\n        \'final_loss\': final[\'valid_loss\'],\n        \'final_token_acc\': final[\'valid_token_acc\'],\n        \'final_exact_match\': final[\'valid_exact_match\'],\n        \'success_metric_name\': task.success_metric,\n        \'success_metric_value\': metric,\n        \'success_threshold\': task.success_threshold,\n        \'success_pass\': success,\n        \'max_em_in_window\': meta.get(\'max_em_in_window\', 0.0),\n        \'max_token_acc_window\': meta.get(\'max_token_acc_window\', 0.0),\n        \'exact_match_by_bucket\': final.get(\'valid_exact_match_by_bucket\', {}),\n        \'max_em_by_bucket_in_window\': meta.get(\'max_em_by_bucket_in_window\', {}),\n        \'seconds\': time.time() - t0,\n        \'hist\': hist,\n    }\n    # Save checkpoint for downstream probes (OOD transfer, etc)\n    os.makedirs(f\'outputs/{variant}/cache\', exist_ok=True)\n    ckpt_path = f\'outputs/{variant}/cache/{task.name}_seed{seed}_dm{d_model}_L{n_layers}_ep{task.epochs}.pt\'\n    torch.save({\'model\': model.state_dict(), \'meta\': meta}, ckpt_path)\n    return ret\n\n\ndef _check_vs_baseline(summary: Dict, baseline_path: str,\n                       epsilon: float = 0.0) -> Dict:\n    """Compara el resumen del run actual contra el snapshot del transformer.\n\n    Lee ``outputs/transformer/benchmark.json`` si existe; si no, devuelve\n    ``baseline_present=False`` y el caller cae a la regla simple\n    (passed >= 3/4).\n\n    Para cada task, ``passes_baseline[task]`` es True si\n    ``own_exact_match >= transformer_exact_match - epsilon`` (epsilon=0.0\n    por default: el proto debe igualar o superar al transformer).\n\n    ``recommend_kaggle`` final:\n        passed >= 3/4 AND (baseline ausente OR passes_baseline en >=2 tasks)\n    """\n    import json as _json\n    import os as _os\n    if not _os.path.exists(baseline_path):\n        return {\n            \'baseline_present\': False,\n            \'baseline_path\': baseline_path,\n            \'epsilon\': epsilon,\n            \'passes_baseline\': None,\n            \'n_passes_baseline\': 0,\n            \'note\': \'snapshot baseline no encontrado; cae a regla passed>=3/4\',\n        }\n    with open(baseline_path, \'r\', encoding=\'utf-8\') as f:\n        baseline = _json.load(f)\n    passes = {}\n    for tname, tr in summary[\'tasks\'].items():\n        own = tr[\'success_metric_value\']\n        base = baseline[\'tasks\'].get(tname, {}).get(\'exact_match\')\n        if base is None:\n            passes[tname] = None\n        else:\n            passes[tname] = bool(own >= base - epsilon)\n    n_passes = sum(1 for v in passes.values() if v is True)\n    n_total = sum(1 for v in passes.values() if v is not None)\n    return {\n        \'baseline_present\': True,\n        \'baseline_path\': baseline_path,\n        \'epsilon\': epsilon,\n        \'passes_baseline\': passes,\n        \'n_passes_baseline\': n_passes,\n        \'n_comparable_tasks\': n_total,\n        \'baseline_variant\': baseline.get(\'variant\'),\n        \'baseline_smoke\': baseline.get(\'smoke\'),\n        \'baseline_seed\': baseline.get(\'seed\'),\n        \'baseline_d_model\': baseline.get(\'d_model\'),\n        \'baseline_n_layers\': baseline.get(\'n_layers\'),\n    }\n\n\ndef run_smoke(variant: str, build_fn: Callable, device, seed: int = 42,\n              lr: float = 1e-3, weight_decay: float = 0.01, clip: float = 1.0,\n              d_model: int = 128, n_layers: int = 2, smoke=True,\n              tasks: Optional[str] = None, epochs_override: int = -1,\n              compile_model: bool = False,\n              optimizer: str = \'adamw\',\n              include_forget: bool = True,\n              include_tiny_program: bool = False,\n              include_dyck2: bool = False,\n              evidence_level: str = \'N1\',\n              loss_fn: Optional[Callable] = None,\n              ss_fn: Optional[Callable] = None) -> Dict:\n    # include_forget (S45): anade la 5a task `forget_retrieval` al harness.\n    # Default True desde 2026-08-03; pasar False para modo legacy 4-tasks\n    # (regla recommend_kaggle passed >= 3 exacto). Con 5 tasks, threshold\n    # de passed sube a n_total - 2 (== 4/5) para preservar ratio 3/4.\n    # include_tiny_program (S61): la 6ta task `tiny_program` es OP-TIN\n    # (default False) para calibrar sin invalidar el baseline historico.\n    # Ademas se auto-activa si `tasks` la nombra explicitamente\n    # (--tasks tiny_program): sin runners modificados, sin inclusion\n    # accidental en otros runs. d3 gatea recommend_kaggle via\n    # bucket_thresholds del TaskSpec (ver mas abajo).\n    # include_dyck2 (S81): la 7ma task `dyck2` es OP-TIN (default False),\n    # misma mecanica que tiny_program (auto-activa si --tasks la nombra);\n    # el bucket d3+ gatea la profundidad de stack (memoria de composicion).\n    from dataclasses import replace\n    tiny_requested = bool(tasks) and \'tiny_program\' in {t.strip() for t in tasks.split(\',\')}\n    dyck2_requested = bool(tasks) and \'dyck2\' in {t.strip() for t in tasks.split(\',\')}\n    specs = make_task_specs(smoke=smoke, include_forget=include_forget,\n                            include_tiny_program=include_tiny_program or tiny_requested,\n                            include_dyck2=include_dyck2 or dyck2_requested)\n    if epochs_override > 0:\n        specs = [replace(s, epochs=epochs_override) for s in specs]\n    if tasks and tasks != \'all\':\n        wanted = set(t.strip() for t in tasks.split(\',\'))\n        specs = [s for s in specs if s.name in wanted]\n    print(f"Reasoning smoke  |  variant={variant}  device={device}  smoke={smoke}  seed={seed}")\n    print(f"Tasks: {[s.name for s in specs]}  d_model={d_model}  n_layers={n_layers}  opt={optimizer}")\n    print("=" * 70)\n    results = []\n    for spec in specs:\n        print(f"  - {spec.name:<16} epochs={spec.epochs} samples={spec.train_samples}/{spec.valid_samples}")\n        res = run_one_task(spec, build_fn, device, seed=seed, lr=lr,\n                           weight_decay=weight_decay, clip=clip,\n                           d_model=d_model, n_layers=n_layers, variant=variant,\n                           compile_model=compile_model, optimizer=optimizer,\n                           loss_fn=loss_fn, ss_fn=ss_fn)\n        results.append(res)\n        print(f"    -> {res[\'success_metric_name\']}={res[\'success_metric_value\']:.3f} "\n              f"target={spec.success_threshold:.2f} pass={res[\'success_pass\']} "\n              f"params={res[\'n_params\']} t={res[\'seconds\']:.1f}s")\n    passed = sum(1 for r in results if r[\'success_pass\'])\n    # --- Comparacion contra baseline del transformer (snapshot) ---\n    # Si el snapshot existe (lo genera transformer_smoke.py --save-baseline),\n    # recommend_kaggle = passed>=3/4 AND passes_baseline en >=2 tasks.\n    # Si no existe, cae a la regla simple passed>=3/4.\n    # S45: con la 5a task activa, threshold de passed sube a n_total-1\n    # (preserva "falla solo 1" como bar). Es decir: 4 tasks siguen\n    # requiriendo 3 pasadas; 5 tasks requieren 4 pasadas (incluida o no\n    # la critica forget_retrieval). Si include_forget=False o --tasks\n    # omiten forget_retrieval, n_total=4 y se sigue el rule anterior\n    # exacto (n_total-1 == 3).\n    import os as _os\n    repo_root = _os.path.dirname(_os.path.dirname(_os.path.abspath(__file__)))\n    baseline_path = _os.path.join(repo_root, \'outputs\', \'transformer\',\n                                  \'benchmark.json\')\n    vs_base = _check_vs_baseline(\n        {\'tasks\': {r[\'task\']: {\'success_metric_value\': r[\'success_metric_value\']}\n                   for r in results}},\n        baseline_path, epsilon=0.0)\n    n_total = len(results)\n    pass_threshold = max(3, n_total - 1)  # 4 -> 3, 5 -> 4, 6 -> 5\n    # S61: gate por bucket (d3 de tiny_program). recommend_kaggle exige que\n    # CADA bucket_threshold del spec se cumpla, ademas de la regla clasica:\n    # fallar tiny_program (o su bucket d3) bloquea la promocion aunque otra\n    # task secundaria falle.\n    spec_by_name = {s.name: s for s in specs}\n    thresholds_by_name = {s.name: s.bucket_thresholds for s in specs}\n    bucket_gates = {}\n    bucket_gates_ok = True\n    for sname, spec in spec_by_name.items():\n        if not spec.bucket_thresholds:\n            continue\n        res = next((r for r in results if r[\'task\'] == sname), None)\n        for k, thr in spec.bucket_thresholds.items():\n            val = (res or {}).get(\'exact_match_by_bucket\', {}).get(k, 0.0)\n            ok = val >= thr\n            bucket_gates[\'%s[%s]\' % (sname, k)] = {\n                \'value\': val, \'threshold\': thr, \'pass\': ok}\n            bucket_gates_ok = bucket_gates_ok and ok\n    if vs_base[\'baseline_present\']:\n        recommend = (passed >= pass_threshold\n                     and vs_base[\'n_passes_baseline\'] >= 2 and bucket_gates_ok)\n    else:\n        recommend = passed >= pass_threshold and bucket_gates_ok\n    # Pin de entorno + git hash (S35.5: PyTorch upgrade silencioso\n    # previene sorpresas; el JSON debe poder replicarse o diff-erse).\n    import platform as _plat\n    import subprocess as _sp\n    try:\n        env = {\n            \'python\': _plat.python_version(),\n            \'torch\': torch.__version__,\n            \'cuda_available\': bool(torch.cuda.is_available()),\n            \'platform\': _plat.platform(),\n        }\n    except Exception as _e:\n        env = {\'error\': str(_e)}\n    git_hash = None\n    try:\n        repo = _os.path.dirname(_os.path.dirname(_os.path.abspath(__file__)))\n        out = _sp.run([\'git\', \'rev-parse\', \'HEAD\'], cwd=repo,\n                      capture_output=True, text=True, timeout=5,\n                      shell=True)\n        if out.returncode == 0:\n            git_hash = out.stdout.strip() or None\n    except Exception:\n        git_hash = None\n    # Propaga max_em_in_window al summary por task\n    return {\n        \'variant\': variant, \'device\': str(device), \'smoke\': smoke, \'seed\': seed,\n        \'lr\': lr, \'weight_decay\': weight_decay, \'clip\': clip,\n        \'optimizer\': optimizer,\n        \'d_model\': d_model, \'n_layers\': n_layers,\n        \'n_params\': results[0][\'n_params\'] if results else 0,\n        \'env\': env,\n        \'git_hash\': git_hash,\n        \'evidence_level\': evidence_level,\n        \'tasks\': {r[\'task\']: {\n            \'epochs\': r[\'epochs\'], \'train_samples\': r[\'train_samples\'],\n            \'valid_samples\': r[\'valid_samples\'],\n            \'final_loss\': r[\'final_loss\'], \'final_token_acc\': r[\'final_token_acc\'],\n            \'final_exact_match\': r[\'final_exact_match\'],\n            \'max_em_in_window\': r.get(\'max_em_in_window\', 0.0),\n            \'max_token_acc_window\': r.get(\'max_token_acc_window\', 0.0),\n            \'exact_match_by_bucket\': r.get(\'exact_match_by_bucket\', {}),\n            \'max_em_by_bucket_in_window\': r.get(\'max_em_by_bucket_in_window\', {}),\n            \'bucket_thresholds\': thresholds_by_name.get(r[\'task\']),\n            \'success_metric_name\': r[\'success_metric_name\'],\n            \'success_metric_value\': r[\'success_metric_value\'],\n            \'success_threshold\': r[\'success_threshold\'],\n            \'success_pass\': r[\'success_pass\'], \'seconds\': r[\'seconds\'],\n            \'hist\': r[\'hist\'],\n        } for r in results},\n        \'passed_count\': passed,\n        \'total_tasks\': len(results),\n        \'recommend_kaggle\': recommend,\n        \'bucket_gates\': bucket_gates,\n        \'vs_baseline\': vs_base,\n    }\n\n\ndef print_summary(summary: Dict):\n    print("\\n" + "=" * 70)\n    print(f"{\'task\':<18}{\'metric\':<12}{\'value\':>8}{\'target\':>8}{\'pass\':>6}{\'time\':>8}")\n    print("-" * 70)\n    for tname, tr in summary[\'tasks\'].items():\n        print(f"{tname:<18}{tr[\'success_metric_name\']:<12}"\n              f"{tr[\'success_metric_value\']:8.3f}{tr[\'success_threshold\']:8.2f}"\n              f"{\'  OK\' if tr[\'success_pass\'] else \'FAIL\':>6}{tr[\'seconds\']:7.1f}s")\n        b = tr.get(\'exact_match_by_bucket\') or {}\n        if b:\n            print("  buckets: " + " ".join(\n                f"{k}={v:.3f}" for k, v in sorted(b.items())))\n    bg = summary.get(\'bucket_gates\') or {}\n    if bg:\n        print("-" * 70)\n        for k, g in bg.items():\n            print(f"  gate {k:<16} {g[\'value\']:8.3f} >= {g[\'threshold\']:.2f}"\n                  f"{\'  OK\' if g[\'pass\'] else \'FAIL\'}")\n    vsb = summary.get(\'vs_baseline\') or {}\n    if vsb.get(\'baseline_present\'):\n        print("\\n" + "-" * 70)\n        print(f"vs transformer baseline (epsilon={vsb[\'epsilon\']:.2f}):")\n        for tname, ok in vsb[\'passes_baseline\'].items():\n            mark = \'==\' if ok is True else (\'--\' if ok is None else \'<<\')\n            print(f"  {tname:<18} {mark}")\n        print(f"  -> {vsb[\'n_passes_baseline\']}/{vsb[\'n_comparable_tasks\']} >= baseline")\n    else:\n        print("\\n" + "-" * 70)\n        print(f"(sin baseline del transformer: {vsb.get(\'note\', \'\')})")\n    print(f"\\n>> {summary[\'variant\']}: {summary[\'passed_count\']}/{summary[\'total_tasks\']} "\n          f"pasadas -> recommend_kaggle={summary[\'recommend_kaggle\']}")\n''')
print('wrote tests/common_smoke.py:', os.path.getsize(os.path.join(base, 'tests/common_smoke.py')))

# --- tests/muon_opt.py ---
with open(os.path.join(base, 'tests/muon_opt.py'), 'w', encoding='utf-8') as f:
    f.write('''"""Muon (Moonlight) - optimizador para el harness de smokes.\n\nImplementacion del Muon con el ajuste de escala de Moonlight (Liu et al.\n2025, arXiv:2502.16982): sobre matrices 2D del transformer se aplica\nmomentum Nesterov + ortogonalizacion Newton-Schulz (5 iters) con rescale\n0.2*sqrt(max(A,B)), y decay aplicado en la actualizacion (W -= lr*(O +\nlambda*W)). Embeddings, head de salida y parametros 1D/0D se optimizan\ncon AdamW (recomendacion del paper: las capas de entrada/salida NO van\npor Muon).\n\nCon el rescale de Moonlight se pueden reusar los mismos lr/wd que AdamW\nsin re-tuning. Costo del Newton-Schulz en matrices chicas (<=128x128,\nnuestro caso) es despreciable frente al paso.\n\nReferencias: Keller Jordan et al. (Muon, 2024); Moonlight (2025) formula\n(4): W_t = W_{t-1} - eta_t (0.2 * O_t * sqrt(max(A,B)) + lambda W_{t-1}).\n"""\n\nimport torch\nimport torch.nn as nn\n\n\ndef zeropower_via_newtonschulz5(G, steps: int = 5, eps: float = 1e-7):\n    """Ortogonalizacion Newton-Schulz de 5 pasos (coeficientes KJ)."""\n    a, b, c = (3.4445, -4.7750, 2.0315)\n    X = G.float()\n    if G.size(0) > G.size(1):\n        X = X.T\n    X = X / (X.norm() + eps)  # top singular value <= 1\n    for _ in range(steps):\n        A = X @ X.T\n        B = b * A + c * A @ A\n        X = a * X + B @ X\n    if G.size(0) > G.size(1):\n        X = X.T\n    return X.to(G.dtype)\n\n\ndef split_params(model):\n    """muon_params: matrices 2D de nn.Linear (excepto el head final).\n    adam_params: embeddings, head, biases y todo parametro 1D/0D."""\n    muon, adam = [], []\n    for mname, mod in model.named_modules():\n        if isinstance(mod, nn.Embedding):\n            adam.append(mod.weight)\n        elif isinstance(mod, nn.Linear):\n            if mname == \'head\' or mname.endswith(\'.head\'):\n                adam.append(mod.weight)\n            else:\n                muon.append(mod.weight)\n    covered = set(id(p) for p in (muon + adam))\n    for p in model.parameters():\n        if id(p) not in covered:\n            adam.append(p)\n    return muon, adam\n\n\nclass Muon(torch.optim.Optimizer):\n    """Muon + AdamW combinados (el paso llama a ambos)."""\n\n    def __init__(self, model, lr: float = 1e-3, momentum: float = 0.95,\n                 weight_decay: float = 0.01, ns_steps: int = 5,\n                 adam_betas=(0.9, 0.999), adam_eps: float = 1e-8):\n        super().__init__([{\'params\': []}], dict(lr=lr, momentum=momentum,\n                                                weight_decay=weight_decay))\n        self.ns_steps = ns_steps\n        self.muon_groups, adam_params = split_params(model)\n        self.muon_mom = [torch.zeros_like(p) for p in self.muon_groups]\n        fused = bool(adam_params) and all(p.is_cuda for p in adam_params)\n        self.adam = torch.optim.AdamW(adam_params, lr=lr,\n                                      weight_decay=weight_decay,\n                                      betas=adam_betas, eps=adam_eps,\n                                      fused=fused)\n\n    @torch.no_grad()\n    def step(self, closure=None):\n        lr = self.defaults[\'lr\']\n        momentum = self.defaults[\'momentum\']\n        wd = self.defaults[\'weight_decay\']\n        for p, buf in zip(self.muon_groups, self.muon_mom):\n            if p.grad is None:\n                continue\n            g = p.grad\n            buf.mul_(momentum).add_(g)\n            g = buf + momentum * g                  # Nesterov\n            O = zeropower_via_newtonschulz5(g, self.ns_steps)\n            O.mul_(0.2 * (max(p.shape) ** 0.5))     # Moonlight rescale\n            p.add_(O, alpha=-lr)\n            p.mul_(1.0 - wd * lr)                   # weight decay\n        self.adam.step()\n\n    def zero_grad(self, set_to_none: bool = True):\n        for p in self.muon_groups:\n            if p.grad is not None:\n                if set_to_none:\n                    p.grad = None\n                else:\n                    p.grad.zero_()\n        self.adam.zero_grad(set_to_none=set_to_none)\n\n\ndef make_optimizer(model, name: str = \'adamw\', lr: float = 1e-3,\n                   weight_decay: float = 0.01):\n    if name == \'adamw\':\n        fused = all(p.is_cuda for p in model.parameters())\n        return torch.optim.AdamW(model.parameters(), lr=lr,\n                                 weight_decay=weight_decay, fused=fused)\n    if name == \'muon\':\n        return Muon(model, lr=lr, weight_decay=weight_decay)\n    raise ValueError(f\'optimizador desconocido: {name}\')\n''')
print('wrote tests/muon_opt.py:', os.path.getsize(os.path.join(base, 'tests/muon_opt.py')))

# --- tests/wave_mem_n1.py ---
with open(os.path.join(base, 'tests/wave_mem_n1.py'), 'w', encoding='utf-8') as f:
    f.write('''"""Runner N1 (O03): 5-task cpu_quick, 3 brazos x 3 seeds, con la spec de\npotencia del auditor (forget_retrieval n_pairs [16,24], >=300 eventos\nFORGET/brazo/seed) + probe de selectividad sobre modelos entrenados con la\nintervencion oracle de re-presentacion + TOST del control + probe de\ndecodificabilidad (ridge sobre M post-erase y sobre el residual de lectura).\nPre-registro: logs/O03_test.md + paper/CLAIM.md §8 items 14-16.\nSalida unica: outputs/wave_mem/n1.json.\n"""\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport time\n\nimport torch\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nREPO = os.path.dirname(HERE)\nsys.path.insert(0, REPO)\n\nfrom dataclasses import replace\n\nfrom tests.common_smoke import (BOS_ID, SEP_ID, QUERY_ID, FORGET_ID,\n                                ANSWER_ID, K_OFFSET_MQAR, V_OFFSET_MQAR,\n                                N_KEYS_MQAR, N_VALS_MQAR, PAD_ID,\n                                ForgetRetrieveDataset, make_task_specs,\n                                run_one_task, evaluate)\nfrom prototypes.wave_mem.model import WaveMemLM\nfrom prototypes.delta_forget.model import DeltaForgetLM\n\nT_CRIT_90_DF2 = 2.920  # t(0.95, 2) para IC 90% con 3 seeds\n\n\ndef gen_fr_query(rng, n_pairs, q_kind):\n    """Misma estructura que gen_forget_retrieve_sample (1 forget) pero con\n    la query dirigida a clave ALIVE o a la clave BORRADA (para fuga)."""\n    ids_k = rng.sample(range(N_KEYS_MQAR), n_pairs)\n    ids_v = rng.sample(range(N_VALS_MQAR), n_pairs)\n    erased = rng.randrange(n_pairs)\n    remaining = [i for i in range(n_pairs) if i != erased]\n    qi = erased if q_kind == \'erased\' else rng.choice(remaining)\n    pair_order = list(range(n_pairs))\n    rng.shuffle(pair_order)\n    store = []\n    for p in pair_order:\n        store += [K_OFFSET_MQAR + ids_k[p], V_OFFSET_MQAR + ids_v[p]]\n    seq = ([BOS_ID] + store +\n           [SEP_ID, FORGET_ID, K_OFFSET_MQAR + ids_k[erased],\n            SEP_ID, QUERY_ID, K_OFFSET_MQAR + ids_k[qi], ANSWER_ID])\n    return seq, V_OFFSET_MQAR + ids_v[qi], V_OFFSET_MQAR + ids_v[erased]\n\n\ndef make_fr_probe_dataset(n_samples, seed, n_pairs_range=(16, 24), kind=\'alive\'):\n    rng = random.Random(seed)\n    seqs, tgts, v_erased = [], [], []\n    for _ in range(n_samples):\n        n_pairs = rng.randint(*n_pairs_range)\n        seq, tgt, ve = gen_fr_query(rng, n_pairs, kind)\n        seqs.append(seq)\n        tgts.append(tgt)\n        v_erased.append(ve)\n    T = max(len(s) for s in seqs)\n    x = torch.full((len(seqs), T), PAD_ID, dtype=torch.long)\n    for i, s in enumerate(seqs):\n        x[i, :len(s)] = torch.tensor(s, dtype=torch.long)\n    y = torch.tensor(tgts, dtype=torch.long)\n    return x, y, seqs, torch.tensor(v_erased, dtype=torch.long)\n\n\ndef build_wave(read_proj):\n    def f(vocab_size, max_len, d_model, n_layers):\n        return WaveMemLM(vocab_size, max_len, d_model, n_layers,\n                         read_proj=read_proj)\n    return f\n\n\ndef build_delta(vocab_size, max_len, d_model, n_layers):\n    return DeltaForgetLM(vocab_size, max_len, d_model, n_layers)\n\n\ndef power_specs():\n    """cpu_quick 5-task con forget_retrieval bajo la spec del auditor:\n    n_pairs [16,24], n_forget [1,2], max_seq_len 64."""\n    specs = make_task_specs(smoke=\'cpu_quick\')\n    out = []\n    for s in specs:\n        if s.name == \'forget_retrieval\':\n            s = replace(\n                s, max_seq_len=64,\n                make_train=lambda seed: ForgetRetrieveDataset(\n                    600, seed=seed, n_pairs_range=(16, 24), n_forget_range=(1, 2)),\n                make_valid=lambda seed: ForgetRetrieveDataset(\n                    150, seed=seed + 1, n_pairs_range=(16, 24), n_forget_range=(1, 2)))\n        out.append(s)\n    return out\n\n\ndef v_true_for(model, tok_v, idx=None):\n    """Valor escrito por el modelo para el token de valor tok_v, por bloque."""\n    vts = []\n    if isinstance(model, WaveMemLM):\n        emb = model.embedding(tok_v)\n        for b in model.blocks:\n            p = b.v_proj(emb)\n            vts.append(torch.complex(p, torch.zeros_like(p)))\n    else:\n        emb = model.embedding(tok_v)\n        for b in model.blocks:\n            vts.append(b.v_proj(emb))\n    return vts\n\n\ndef key_for(model, tok_k, idx=None):\n    if isinstance(model, WaveMemLM):\n        return model.codebook[tok_k]\n    ks = []\n    emb = model.embedding(tok_k)\n    for b in model.blocks:\n        ks.append(b.k_proj(emb))\n    return ks\n\n\ndef run_probe(model, x, y, seqs, tok_v_forget, mode, device):\n    """Forward con posible intervencion oracle de re-presentacion.\n    mode: \'reread\' (sin intervencion: el modelo borra por re-lectura) |\n          \'represent\' (el evaluador borra con el v VERDADERO inyectado).\n    Captura por muestra: EM del target (y), readout de la clave consultada\n    (por bloque) y M post-erase del bloque 0.\n    """\n    B, T = x.shape\n    tok = x\n    pos_forget = torch.tensor([s.index(FORGET_ID) for s in seqs], dtype=torch.long)\n    pos_qkey = torch.tensor([s.index(QUERY_ID) + 1 for s in seqs], dtype=torch.long)\n    pos_ans = torch.tensor([s.index(ANSWER_ID) for s in seqs], dtype=torch.long)\n    tok_key = torch.tensor([s[int(pos_forget[i]) + 1] for i, s in enumerate(seqs)], dtype=torch.long)\n    is_delta = isinstance(model, DeltaForgetLM)\n    rb_dtype = torch.float32 if is_delta else torch.complex64\n    state = model.init_state(B, device)\n    readout_b = [torch.zeros(B, model.d_model, dtype=rb_dtype) for _ in range(model.n_layers)]\n    M_post = torch.zeros(B, model.d_model, model.d_model,\n                         dtype=torch.float32 if is_delta else torch.complex64)\n    pred = torch.zeros(B, dtype=torch.long)\n    with torch.no_grad():\n        for t in range(T):\n            if mode == \'represent\':\n                m = (pos_forget == t - 1)\n                if m.any():\n                    idx = m.nonzero(as_tuple=False).squeeze(-1)\n                    vv = tok_v_forget[idx]\n                    ks = key_for(model, tok[idx, t])\n                    vts = v_true_for(model, vv)\n                    for b in range(model.n_layers):\n                        St = state.S if is_delta else state.M\n                        M_b = St[b]\n                        M_new = M_b.clone()\n                        k = ks if not is_delta else ks[b]\n                        delta = -k.unsqueeze(-1) * vts[b].unsqueeze(1)\n                        M_new[idx] = M_b[idx] + delta\n                        if is_delta:\n                            state.S[b] = M_new\n                        else:\n                            state.M[b] = M_new\n                    prev = state.prev.clone()\n                    prev[idx] = tok[idx, t]\n                    state.prev = prev\n            logits_t, state = model.decode_step(tok[:, t], state)\n            if (pos_qkey == t).any():\n                qm = pos_qkey == t\n                for b in range(model.n_layers):\n                    rb = readout_b[b].clone()\n                    rb[qm] = state.r[b][qm]\n                    readout_b[b] = rb\n            if (pos_forget == t - 1).any():\n                pm = pos_forget == t - 1\n                mp = M_post.clone()\n                St = state.S if is_delta else state.M\n                mp[pm] = St[0][pm]\n                M_post = mp\n            if (pos_ans == t).any():\n                am = pos_ans == t\n                p = pred.clone()\n                p[am] = logits_t[am].argmax(dim=-1)\n                pred = p\n    tgts = y.to(torch.long)\n    em = float((pred == tgts).float().mean().item())\n    return em, readout_b, M_post\n\n\ndef state_fuga(model, readout, tok_v_erased, device):\n    """|r|^2/|v|^2 del readout de la clave borrada, por bloque."""\n    vts = v_true_for(model, torch.tensor(tok_v_erased, dtype=torch.long, device=device))\n    vals = []\n    for b in range(model.n_layers):\n        r = readout[b]\n        v = vts[b]\n        num = r.abs().pow(2).sum(dim=1)\n        den = v.abs().pow(2).sum(dim=1).clamp(min=1e-12)\n        vals.append(float((num / den).mean().item()))\n    return vals\n\n\ndef ridge_probe(X_tr, y_tr, X_te, y_te, lam=0.1):\n    """Ridge lineal: features -> target real. Reporta cosine medio test."""\n    Xtr = X_tr - X_tr.mean(dim=0, keepdim=True)\n    Xte = X_te - X_te.mean(dim=0, keepdim=True)\n    ytr = y_tr - y_tr.mean(dim=0, keepdim=True)\n    yte = y_te - y_te.mean(dim=0, keepdim=True)\n    f = Xtr.size(1)\n    XtX = Xtr.t() @ Xtr + lam * torch.eye(f, dtype=Xtr.dtype)\n    W = torch.linalg.solve(XtX, Xtr.t() @ ytr)\n    yp = Xte @ W\n    num = (yp * yte).sum(dim=1)\n    den = yp.norm(dim=1) * yte.norm(dim=1)\n    cos = float((num / den.clamp(min=1e-12)).mean().item())\n    ss_res = float((yp - yte).pow(2).sum(dim=1).mean().item())\n    ss_tot = float(yte.pow(2).sum(dim=1).mean().item())\n    return {\'cosine_test\': round(cos, 4), \'mse_rel\': round(ss_res / max(ss_tot, 1e-12), 4)}\n\n\ndef probe_features(model, readout, M_post, device):\n    """Features de ridge: readout residual del bloque 0 (lo que ve el head)\n    y proyeccion aleatoria fija de M post-erase (lo que podria ver un\n    decodificador lineal sobre M)."""\n    if isinstance(model, WaveMemLM):\n        if model.read_proj == \'complex\':\n            feat_r = torch.cat([readout[0].real, readout[0].imag], dim=1)\n        else:\n            feat_r = readout[0].real\n    else:\n        feat_r = readout[0]\n    if M_post.is_complex():\n        Mf = torch.cat([M_post.real.flatten(1), M_post.imag.flatten(1)], dim=1)\n    else:\n        Mf = M_post.flatten(1)\n    W = torch.randn(Mf.size(1), 64, generator=torch.Generator().manual_seed(99))\n    feat_m = Mf @ W\n    return feat_r.to(torch.float32), feat_m.to(torch.float32)\n\n\ndef tost_equivalence(diffs, eps=0.02):\n    """TOST: |media| + t(0.95, df)*SE <= eps con IC 90%."""\n    n = len(diffs)\n    mu = sum(diffs) / n\n    sd = (sum((d - mu) ** 2 for d in diffs) / max(n - 1, 1)) ** 0.5\n    se = sd / math.sqrt(n)\n    half = T_CRIT_90_DF2 * se\n    return {\'mean_delta\': round(mu, 4), \'sd\': round(sd, 4),\n            \'ci90\': [round(mu - half, 4), round(mu + half, 4)],\n            \'equivalence_pass\': bool(abs(mu) + half <= eps), \'eps\': eps}\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--device\', type=str, default=\'cpu\')\n    ap.add_argument(\'--d_model\', type=int, default=64)\n    ap.add_argument(\'--n_layers\', type=int, default=2)\n    ap.add_argument(\'--seeds\', type=str, default=\'1,2,3\')\n    ap.add_argument(\'--n_probe\', type=int, default=320)\n    ap.add_argument(\'--out\', type=str, default=\'outputs/wave_mem/n1.json\')\n    args = ap.parse_args()\n    device = torch.device(args.device)\n    seeds = [int(s) for s in args.seeds.split(\',\')]\n\n    arms = [(\'wave_complex\', build_wave(\'complex\')),\n            (\'wave_re\', build_wave(\'re\')),\n            (\'delta_forget\', build_delta)]\n    specs = power_specs()\n\n    t0 = time.time()\n    out = {\'variant\': \'wave_mem_n1\', \'d_model\': args.d_model,\n           \'n_layers\': args.n_layers, \'seeds\': seeds,\n           \'n_probe_events\': args.n_probe,\n           \'spec\': \'cpu_quick 5-task, FR n_pairs [16,24] n_forget [1,2]\',\n           \'runs\': {}}\n    tost_pool = []\n    for seed in seeds:\n        for arm_name, build in arms:\n            label = f\'{arm_name}_s{seed}\'\n            print(f\'\\n===== N1 {arm_name} seed={seed} =====\', flush=True)\n\n            # Resume: si los 5 ckpts del combo ya existen, no re-entrenar.\n            def _ckpt(spec):\n                return (f\'outputs/n1_{arm_name}/cache/{spec.name}_seed{seed}\'\n                        f\'_dm{args.d_model}_L{args.n_layers}_ep{spec.epochs}.pt\')\n            resume = all(os.path.exists(_ckpt(s)) for s in specs)\n            if resume:\n                print(\'  resume: ckpts encontrados, reuso entrenamiento\', flush=True)\n                runs = {}\n                for s in specs:\n                    ck = torch.load(_ckpt(s), map_location=device)\n                    fin = ck[\'meta\'][\'final\']\n                    runs[s.name] = {\n                        \'final_exact_match\': fin[\'valid_exact_match\'],\n                        \'final_loss\': fin[\'valid_loss\'],\n                        \'final_token_acc\': fin[\'valid_token_acc\'],\n                        \'max_em_in_window\': ck[\'meta\'].get(\'max_em_in_window\', 0.0),\n                        \'seconds\': ck[\'meta\'].get(\'total_seconds\', 0.0)}\n                    print(f\'  (resume) {s.name:<16} EM={runs[s.name]["final_exact_match"]:.3f}\',\n                          flush=True)\n            else:\n                runs = {}\n                for spec in specs:\n                    res = run_one_task(spec, build, device, seed=seed, lr=1e-3,\n                                       weight_decay=0.01, clip=1.0,\n                                       d_model=args.d_model, n_layers=args.n_layers,\n                                       variant=f\'n1_{arm_name}\')\n                    runs[spec.name] = {\'final_exact_match\': res[\'final_exact_match\'],\n                                       \'final_loss\': res[\'final_loss\'],\n                                       \'final_token_acc\': res[\'final_token_acc\'],\n                                       \'max_em_in_window\': res.get(\'max_em_in_window\', 0.0),\n                                       \'seconds\': res[\'seconds\']}\n                    print(f\'  {spec.name:<16} EM={res["final_exact_match"]:.3f} \'\n                          f\'loss={res["final_loss"]:.3f} t={res["seconds"]:.0f}s\', flush=True)\n            ckpt = _ckpt(next(s for s in specs if s.name == \'forget_retrieval\'))\n            model = build(vocab_size=89, max_len=64, d_model=args.d_model,\n                          n_layers=args.n_layers)\n            ck = torch.load(ckpt, map_location=device)\n            model.load_state_dict(ck[\'model\'])\n            model.to(device).eval()\n\n            probe = {}\n            fr_ok = True\n            if any(torch.isnan(p).any().item() for p in model.parameters()):\n                print(f\'  WARNING: ckpt {arm_name} seed {seed} NaN (FR no entrena \'\n                      f\'- probable: regla delta no aprende FR a d=64)\', flush=True)\n                fr_ok = False\n                probe[\'fr_status\'] = \'nan_failed\'\n                out[\'runs\'][f\'{arm_name}_s{seed}\'] = {\'tasks\': runs, \'probe\': probe}\n                continue\n\n            # Autocheck: la EM del probe (alive) debe matchear la EM del\n            # harness en formato oficial (con token de valor tras ANSWER).\n            # Si divergen, el probe esta roto -> aborta este arm/seed.\n            from tests.common_smoke import collate_fn\n            from torch.utils.data import DataLoader\n            from functools import partial\n            val_ds = ForgetRetrieveDataset(64, seed=seed + 100, n_pairs_range=(16, 24),\n                                           n_forget_range=(1, 2))\n            col = partial(collate_fn, answer_marker_id=ANSWER_ID, mark_after_marker=False,\n                          prefix_answer=False)\n            val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=col)\n            val_res = evaluate(model, val_loader, device, vocab_size=89)\n            print(f\'  autocheck harness EM={val_res["exact_match"]:.3f}\', flush=True)\n\n            probe = {}\n            for kind in (\'alive\', \'erased\'):\n                x, y, seqs, v_erased = make_fr_probe_dataset(\n                    args.n_probe, seed=1000 + seed, kind=kind)\n                x = x.to(device)\n                em_rr, rd_rr, m_rr = run_probe(model, x, y, seqs, v_erased,\n                                               \'reread\', device)\n                em_rp, rd_rp, m_rp = run_probe(model, x, y, seqs, v_erased,\n                                               \'represent\', device)\n                if kind == \'alive\':\n                    probe_match = abs(em_rr - val_res[\'exact_match\']) < 0.15\n                    if not probe_match:\n                        print(f\'  WARNING: probe alive EM {em_rr:.3f} no matchea \'\n                              f\'harness {val_res["exact_match"]:.3f} - probe roto\',\n                              flush=True)\n                        sys.exit(2)\n                entry = {\'events\': args.n_probe,\n                         \'em_reread\': round(em_rr, 4),\n                         \'em_represent\': round(em_rp, 4)}\n                if kind == \'erased\':\n                    tok_vs = [int(v) for v in v_erased]\n                    entry[\'fuga_power_reread\'] = [\n                        round(v, 4) for v in state_fuga(model, rd_rr, tok_vs, device)]\n                    entry[\'fuga_power_represent\'] = [\n                        round(v, 4) for v in state_fuga(model, rd_rp, tok_vs, device)]\n                    ytr = v_true_for(model, torch.tensor(tok_vs, dtype=torch.long, device=device))[0]\n                    if isinstance(model, WaveMemLM):\n                        ytr = ytr.real\n                    fr, fm = probe_features(model, rd_rr, m_rr, device)\n                    n_tr = int(len(seqs) * 0.75)\n                    entry[\'ridge_readout\'] = ridge_probe(fr[:n_tr], ytr[:n_tr], fr[n_tr:], ytr[n_tr:])\n                    entry[\'ridge_M_proj\'] = ridge_probe(fm[:n_tr], ytr[:n_tr], fm[n_tr:], ytr[n_tr:])\n                probe[kind] = entry\n            sel_rr = round(probe[\'alive\'][\'em_reread\'] - probe[\'erased\'][\'em_reread\'], 4)\n            sel_rp = round(probe[\'alive\'][\'em_reread\'] - probe[\'erased\'][\'em_represent\'], 4)\n            probe[\'selectividad_reread\'] = sel_rr\n            probe[\'selectividad_represent\'] = sel_rp\n            out[\'runs\'][label] = {\'tasks\': runs, \'probe\': probe}\n            if arm_name in (\'wave_complex\', \'delta_forget\'):\n                tost_pool.append((arm_name, seed, sel_rr))\n            print(f\'  probe: EM_vivo={probe["alive"]["em_reread"]} \'\n                  f\'EM_fuga_reread={probe["erased"]["em_reread"]} \'\n                  f\'sel_reread={sel_rr} sel_represent={sel_rp}\', flush=True)\n\n    wc = {s: next(v for a, ss, v in tost_pool if a == \'wave_complex\' and ss == s) for s in seeds}\n    dl = {s: next((v for a, ss, v in tost_pool if a == \'delta_forget\' and ss == s),\n                  None) for s in seeds}\n    common = [s for s in seeds if dl[s] is not None]\n    if len(common) >= 3:\n        deltas = [wc[s] - dl[s] for s in common]\n        out[\'tost_control\'] = tost_equivalence(deltas)\n    else:\n        out[\'tost_control\'] = {\n            \'status\': \'NO COMPUTABLE en FR\',\n            \'reason\': \'delta_forget no entrena forget_retrieval a d=64 \'\n                      \'(EM ~ chance, loss NaN): el control de identidad \'\n                      \'wave_complex vs delta no existe en FR\',\n            \'n_common\': len(common)}\n    re_by_seed = {}\n    for s in seeds:\n        re_by_seed[s] = out[\'runs\'][f\'wave_re_s{s}\'][\'probe\'][\'selectividad_reread\']\n    out[\'contrast_re_vs_others\'] = {\n        \'by_seed\': {\n            str(s): {\'sel_re\': re_by_seed[s], \'sel_wc\': wc[s],\n                     \'sel_delta\': dl[s] if dl[s] is not None else None}\n            for s in seeds},\n        \'pred_deficit_re\': \'(n-1)/(2D) en fuga del canal re vs complex\',\n    }\n\n    import platform\n    import subprocess\n    out[\'env\'] = {\'python\': platform.python_version(), \'torch\': torch.__version__}\n    try:\n        r = subprocess.run([\'git\', \'rev-parse\', \'HEAD\'], cwd=REPO,\n                           capture_output=True, text=True, shell=True, timeout=5)\n        out[\'git_hash\'] = r.stdout.strip() or None\n    except Exception:\n        out[\'git_hash\'] = None\n    out[\'total_seconds\'] = round(time.time() - t0, 1)\n\n    os.makedirs(os.path.dirname(args.out), exist_ok=True)\n    with open(args.out, \'w\', encoding=\'utf-8\') as f:\n        json.dump(out, f, indent=2, default=str)\n\n    print(\'\\n===JSON_START===\')\n    print(json.dumps({\n        \'tost_control\': out[\'tost_control\'],\n        \'selectividad_por_seed\': out[\'contrast_re_vs_others\'][\'by_seed\'],\n        \'runs\': {k: v[\'probe\'] for k, v in out[\'runs\'].items()},\n    }, indent=2, default=str))\n    print(\'===JSON_END===\')\n    print(f\'Escrito: {args.out}  total={out["total_seconds"]}s\')\n\n\nif __name__ == \'__main__\':\n    main()''')
print('wrote tests/wave_mem_n1.py:', os.path.getsize(os.path.join(base, 'tests/wave_mem_n1.py')))

# --- tests/o04b_run.py ---
with open(os.path.join(base, 'tests/o04b_run.py'), 'w', encoding='utf-8') as f:
    f.write('''"""Runner O04b (reconciliacion 2x2 del auditor): probe de drift\ncanal {complex, re} x drift {clave, estado} x delta sobre modelos wave\nentrenados (sin reentrenar) + verificacion sintetica de las 4 celdas bajo\n2 convenciones de medicion. Pre-registro: logs/O04b_test.md.\n\nInyeccion por celda (explicita, uniforme):\n- clave: items escritos limpios; erase en t=pos_forget+1 lee\n  conj(e^{id}w).M/D (Re truncado en canal re) y remueve e^{id}w (x) wh.\n- estado: SOLO el item olvidado (pending_id == clave olvidada) se escribe\n  con phasor e^{id}w (los demas limpios); el erase lee con conj(e^{id}w)\n  (el drift se cancela en la estimacion) y remueve con la clave LIMPIA w\n  (reloj actual).\n- Readout de la query SIEMPRE con clave limpia (Re truncado en canal re).\n\nConvenciones: \'fiel\' = Re truncado en wh Y en el readout (lo que ve el\nhead entrenado; gate d=128); \'auditor\' = Re solo en wh (readout complejo;\nnumeros del auditor 0.289/0.0870).\n\nSalida: outputs/wave_mem/o04b.json\n"""\n\nimport argparse\nimport json\nimport math\nimport os\nimport sys\nimport time\n\nimport torch\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nREPO = os.path.dirname(HERE)\nsys.path.insert(0, REPO)\n\nfrom tests.common_smoke import ANSWER_ID, FORGET_ID, QUERY_ID\nfrom tests.wave_mem_n1 import build_wave, make_fr_probe_dataset, v_true_for\nfrom prototypes.wave_mem.model import WaveMemLM, MARKER_MIN\n\nDELTAS = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5)\nN_EVENTS = 320\n\n\ndef law(cell, channel, dlt, c, conv):\n    """Leyes cerradas por celda. conv: \'fiel\' | \'auditor\'."""\n    if channel == \'complex\':\n        return 0.0 if cell == \'clave\' else 2 * (1 - math.cos(dlt)) * (1 + c)\n    if conv == \'fiel\':\n        if cell == \'clave\':\n            return math.sin(dlt) ** 4 + c * (1 - math.cos(2 * dlt)) / 4\n        return (1 - math.cos(dlt)) ** 2 + c * (1 - math.cos(dlt))\n    if cell == \'clave\':\n        return math.sin(dlt) ** 2 + c / 2\n    return 2 * (1 - math.cos(dlt)) + c * (3 - 2 * math.cos(dlt)) / 2\n\n\ndef replay_wave(model, x, y, seqs, v_erased, cell, dlt, device, per_event=False):\n    """Replay externo exacto del forward wave (memoria externa, sin\n    decode_step del modelo) con inyeccion por celda. Devuelve EM y fuga\n    POOLED por capa. Con per_event=True devuelve ademas los ratios de\n    potencia residual por evento (capa 0) para la guarda de residual nulo\n    (O05, adicion del auditor)."""\n    B, T = x.shape\n    is_re = model.read_proj == \'re\'\n    D = model.d_model\n    cb = model.codebook\n    ph = torch.complex(torch.tensor(math.cos(dlt)), torch.tensor(math.sin(dlt)))\n    pos_forget = torch.tensor([s.index(FORGET_ID) for s in seqs], dtype=torch.long,\n                              device=device)\n    pos_qkey = torch.tensor([s.index(QUERY_ID) + 1 for s in seqs], dtype=torch.long,\n                            device=device)\n    pos_ans = torch.tensor([s.index(ANSWER_ID) for s in seqs], dtype=torch.long,\n                           device=device)\n    tok_key = torch.tensor([s[int(pos_forget[i]) + 1] for i, s in enumerate(seqs)],\n                           dtype=torch.long, device=device)\n    M = [torch.zeros(B, D, D, dtype=torch.complex64, device=device)\n         for _ in range(model.n_layers)]\n    r_persist = [torch.zeros(B, D, dtype=torch.complex64, device=device)\n                 for _ in range(model.n_layers)]\n    prev = torch.zeros(B, dtype=torch.long, device=device)\n    expect = torch.zeros(B, dtype=torch.bool, device=device)\n    pending = torch.zeros(B, dtype=torch.long, device=device)\n    pred = torch.zeros(B, dtype=torch.long, device=device)\n    with torch.no_grad():\n        for t in range(T):\n            tok = x[:, t]\n            non_marker = tok >= MARKER_MIN\n            write = expect & non_marker\n            erase = (prev == FORGET_ID) & non_marker\n            read = (prev == QUERY_ID) & non_marker\n            if write.any():\n                vts = v_true_for(model, tok)\n                wv = cb[pending]\n                if cell == \'estado\':\n                    fg = (pending == tok_key).unsqueeze(-1)\n                    wv = torch.where(fg, ph * wv, wv)\n                for b in range(model.n_layers):\n                    M[b][write] += wv[write].unsqueeze(-1) * vts[b][write].unsqueeze(1)\n            if erase.any():\n                wd = ph * cb[tok]\n                for b in range(model.n_layers):\n                    wh = torch.einsum(\'bd,bdj->bj\', torch.conj(wd)[erase], M[b][erase]) / D\n                    if is_re:\n                        wh = torch.complex(wh.real, torch.zeros_like(wh.real))\n                    outer = wd if cell == \'clave\' else cb[tok]\n                    M[b][erase] -= outer[erase].unsqueeze(-1) * wh.unsqueeze(1)\n            if read.any():\n                for b in range(model.n_layers):\n                    rr = torch.einsum(\'bd,bdj->bj\', torch.conj(cb[tok])[read], M[b][read]) / D\n                    if is_re:\n                        rr = torch.complex(rr.real, torch.zeros_like(rr.real))\n                    rp = r_persist[b].clone()\n                    rp[read] = rr\n                    r_persist[b] = rp\n            if (pos_ans == t).any():\n                am = pos_ans == t\n                h = model.embedding(tok)\n                for b, block in enumerate(model.blocks):\n                    rb = r_persist[b]\n                    feat = torch.cat([rb.real, rb.imag], dim=-1) if not is_re else rb.real\n                    h = h + block.out_proj(feat)\n                logits = model.head(model.norm(h))\n                p = pred.clone()\n                p[am] = logits[am].argmax(dim=-1)\n                pred = p\n            prev = tok\n            expect = non_marker & ~expect\n            pending = torch.where(non_marker & ~write, tok, pending)\n    em = float((pred == y.to(device)).float().mean().item())\n    vts = v_true_for(model, v_erased.to(device))\n    fuga = []\n    for b in range(model.n_layers):\n        r = r_persist[b]\n        v = vts[b]\n        if is_re:\n            r = r.real\n            v = v.real\n        fuga.append(float(r.abs().pow(2).sum().item() /\n                          v.abs().pow(2).sum().clamp(min=1e-12).item()))\n    if per_event:\n        r0 = r_persist[0]\n        v0 = vts[0]\n        if is_re:\n            r0 = r0.real\n            v0 = v0.real\n        ratios = (r0.abs().pow(2).sum(dim=1) /\n                  v0.abs().pow(2).sum(dim=1).clamp(min=1e-12))\n        return em, fuga, ratios\n    return em, fuga\n\n\ndef synthetic_cell(channel, cell, dlt, D, n, n_events, seed, conv):\n    """Celda sintetica (operador puro, unica capa): fuga pooled."""\n    g = torch.Generator().manual_seed(seed)\n    th = torch.rand(40, D, generator=g) * 2.0 * math.pi\n    cb = torch.complex(torch.cos(th), torch.sin(th))\n    ph = complex(math.cos(dlt), math.sin(dlt))\n    M = torch.zeros(D, D, dtype=torch.complex64)\n    num = den = 0.0\n    for _ in range(n_events):\n        ks = torch.randperm(40, generator=g)[:n]\n        v = torch.randn(n, D, generator=g)\n        M.zero_()\n        for i in range(n):\n            w = cb[int(ks[i].item())]\n            vv = torch.complex(v[i], torch.zeros_like(v[i]))\n            ww = ph * w if (cell == \'estado\' and i == 0) else w\n            M += ww.unsqueeze(1) * vv.unsqueeze(0)\n        we = cb[int(ks[0].item())]\n        wd = ph * we\n        proj = (wd.conj() @ M) / D\n        if channel == \'re\':\n            proj = torch.complex(proj.real, torch.zeros_like(proj.real))\n        outer = wd if cell == \'clave\' else we\n        M = M - outer.unsqueeze(1) * proj.unsqueeze(0)\n        rr = (we.conj() @ M) / D\n        if channel == \'re\' and conv == \'fiel\':\n            rr = torch.complex(rr.real, torch.zeros_like(rr.real))\n        num += rr.abs().pow(2).sum().item()\n        den += v[0].abs().pow(2).sum().item()\n    return num / max(den, 1e-12)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--device\', type=str, default=\'cpu\')\n    ap.add_argument(\'--n_events\', type=int, default=N_EVENTS)\n    ap.add_argument(\'--out\', type=str, default=\'outputs/wave_mem/o04b.json\')\n    args = ap.parse_args()\n    device = torch.device(args.device)\n    seeds = (1, 2, 3)\n    t0 = time.time()\n\n    out = {\'variant\': \'o04b\', \'d_model\': 64, \'n_layers\': 2,\n           \'seeds\': list(seeds), \'deltas\': list(DELTAS),\n           \'n_events_per_combo\': args.n_events,\n           \'convencion\': (\'fiel: Re truncado en wh y readout (lo que ve el \'\n                          \'head entrenado). auditor: Re solo en wh (readout \'\n                          \'complejo) - numeros del auditor 0.289/0.0870\'),\n           \'inyeccion\': {\'clave\': \'items limpios; erase lee y remueve con \'\n                                 \'e^{id}w (reloj drifteado en el FORGET)\',\n                         \'estado\': \'solo el item olvidado escrito con e^{id}w; \'\n                                   \'erase lee con e^{id}w (drift cancelado) y \'\n                                   \'remueve con w limpia (reloj actual)\'},\n           \'synthetic\': {}, \'runs\': {}}\n\n    # 1) Verificacion sintetica: 2 grillas x 2 convenciones x 4 celdas x 6 delta\n    print(\'===== sintetico =====\', flush=True)\n    for (D, n) in ((64, 20), (128, 16)):\n        c = (n - 1) / D\n        for conv in (\'fiel\', \'auditor\'):\n            key = f\'D{D}_n{n}_{conv}\'\n            out[\'synthetic\'][key] = {\'c\': c, \'cells\': {}}\n            for channel in (\'complex\', \'re\'):\n                for cell in (\'clave\', \'estado\'):\n                    rows = {}\n                    for dlt in DELTAS:\n                        fu = synthetic_cell(channel, cell, dlt, D, n,\n                                            args.n_events, 777, conv)\n                        rows[str(dlt)] = {\'fuga\': round(fu, 4),\n                                          \'pred\': round(law(cell, channel, dlt, c, conv), 4)}\n                    out[\'synthetic\'][key][\'cells\'][f\'{channel}/{cell}\'] = rows\n                    print(f\'  {key} {channel}/{cell}: \'\n                          f\'d=0.5 fuga={rows["0.5"]["fuga"]} pred={rows["0.5"]["pred"]}\',\n                          flush=True)\n\n    # 2) Probe sobre modelos entrenados (replay externo, 4 celdas x 6 delta x 6 ckpts)\n    print(\'===== probe modelos entrenados (replay externo) =====\', flush=True)\n    for arm, read_proj in ((\'wave_complex\', \'complex\'), (\'wave_re\', \'re\')):\n        for seed in seeds:\n            label = f\'{arm}_s{seed}\'\n            ck = torch.load(f\'outputs/n1_{arm}/cache/forget_retrieval_seed{seed}\'\n                            f\'_dm64_L2_ep80.pt\', map_location=device)\n            model = build_wave(read_proj)(89, 64, 64, 2)\n            model.load_state_dict(ck[\'model\'])\n            model.to(device).eval()\n            rows = {}\n            for dlt in DELTAS:\n                x, y, seqs, v_erased = make_fr_probe_dataset(\n                    args.n_events, seed=2000 + seed, kind=\'erased\',\n                    n_pairs_range=(16, 24))\n                x = x.to(device)\n                em_c, fu_c = replay_wave(model, x, y, seqs, v_erased, \'clave\', dlt, device)\n                em_e, fu_e = replay_wave(model, x, y, seqs, v_erased, \'estado\', dlt, device)\n                n_pairs_avg = 20.0\n                c = (n_pairs_avg - 1) / 64\n                rows[str(dlt)] = {\n                    \'em_clave\': round(em_c, 4), \'em_estado\': round(em_e, 4),\n                    \'fuga_clave\': [round(v, 4) for v in fu_c],\n                    \'fuga_estado\': [round(v, 4) for v in fu_e],\n                    \'pred_clave\': round(law(\'clave\', read_proj, dlt, c, \'fiel\'), 4),\n                    \'pred_estado\': round(law(\'estado\', read_proj, dlt, c, \'fiel\'), 4),\n                }\n                print(f\'  {label} d={dlt}: fu_clave={rows[str(dlt)]["fuga_clave"][0]} \'\n                      f\'fu_estado={rows[str(dlt)]["fuga_estado"][0]} \'\n                      f\'EM_c={em_c:.2f} EM_e={em_e:.2f}\', flush=True)\n            out[\'runs\'][label] = rows\n\n    out[\'total_seconds\'] = round(time.time() - t0, 1)\n    import platform\n    import subprocess\n    out[\'env\'] = {\'python\': platform.python_version(), \'torch\': torch.__version__}\n    try:\n        r = subprocess.run([\'git\', \'rev-parse\', \'HEAD\'], cwd=REPO,\n                           capture_output=True, text=True, shell=True, timeout=5)\n        out[\'git_hash\'] = r.stdout.strip() or None\n    except Exception:\n        out[\'git_hash\'] = None\n    os.makedirs(os.path.dirname(args.out), exist_ok=True)\n    with open(args.out, \'w\', encoding=\'utf-8\') as f:\n        json.dump(out, f, indent=2, default=str)\n    print(f\'Escrito: {args.out}  total={out["total_seconds"]}s\')\n\n\nif __name__ == \'__main__\':\n    main()''')
print('wrote tests/o04b_run.py:', os.path.getsize(os.path.join(base, 'tests/o04b_run.py')))

# --- tests/wave_mem_smoke128.py ---
with open(os.path.join(base, 'tests/wave_mem_smoke128.py'), 'w', encoding='utf-8') as f:
    f.write('''"""Runner O05 (mini-smoke d=128, AUTORIZADO por el auditor 2026-08-19):\n3 brazos (wave_complex, wave_re, delta_nlms) x 5 seeds, FR con n_pairs\n[32,48] (n/D 0.25-0.38), 150 epocas, probe 2x2 con guarda de residual\nnulo (EM_erased solo si ||residual|| > eps), C1 a c emparejada (probe\nd=64 con n_pairs (17,24) -> c = 0.3047 = c de d=128) y TOST reportado\ntal cual. Pre-registro: logs/O05_test.md (adiciones del auditor).\nSalida unica: outputs/wave_mem/o05.json + o05_run.log.\n"""\n\nimport argparse\nimport json\nimport math\nimport os\nimport sys\nimport time\n\nimport torch\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nREPO = os.path.dirname(HERE)\nsys.path.insert(0, REPO)\nprint(f\'O05 runner: cwd={os.getcwd()} REPO={REPO} path0={sys.path[:3]}\', flush=True)\n\nfrom dataclasses import replace\n\nimport tests.common_smoke as CS\nfrom tests.common_smoke import (ANSWER_ID, FORGET_ID, QUERY_ID,\n                                ForgetRetrieveDataset, collate_fn, evaluate,\n                                run_one_task)\nfrom tests.wave_mem_n1 import build_wave, make_fr_probe_dataset\nfrom tests.o04b_run import law, replay_wave\nfrom prototypes.delta_nlms.model import DeltaNlmsLM\n\nDELTAS = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5)\nN_EVENTS = 320\nEPS_NULL = 1e-4\nT_CRIT_90_DF4 = 2.132  # t(0.95, 4) para IC 90% con 5 seeds\n\n\nclass KeySpace:\n    """Espacio de claves/valores de O05: n_pairs [32,48] excede los 40\n    keys del harness -> 48 keys / 48 vals / vocab 105 (9+48+48) y\n    V_OFFSET=57. Parche en runtime de las constantes del modulo y de las\n    copias que ya importo wave_mem_n1 (gen_fr_query); el harness fuente\n    queda intacto y se restaura al salir. Fuera de este contexto (probe\n    C1 d=64) se usa el espacio original (40/40/vocab 89)."""\n\n    def __init__(self, n_keys, n_vals, vocab, v_offset):\n        self.cs = CS\n        import tests.wave_mem_n1 as N1\n        self.n1 = N1\n        self.saved = (CS.N_KEYS_MQAR, CS.N_VALS_MQAR,\n                      CS.VOCAB_MQAR_SIZE, CS.VOCAB_FORGET_SIZE,\n                      CS.V_OFFSET_MQAR, N1.N_KEYS_MQAR, N1.N_VALS_MQAR,\n                      N1.V_OFFSET_MQAR)\n        self.new = (n_keys, n_vals, vocab, vocab, v_offset,\n                    n_keys, n_vals, v_offset)\n\n    def __enter__(self):\n        (self.cs.N_KEYS_MQAR, self.cs.N_VALS_MQAR,\n         self.cs.VOCAB_MQAR_SIZE, self.cs.VOCAB_FORGET_SIZE,\n         self.cs.V_OFFSET_MQAR, self.n1.N_KEYS_MQAR, self.n1.N_VALS_MQAR,\n         self.n1.V_OFFSET_MQAR) = self.new\n        return self.cs\n\n    def __exit__(self, *a):\n        (self.cs.N_KEYS_MQAR, self.cs.N_VALS_MQAR,\n         self.cs.VOCAB_MQAR_SIZE, self.cs.VOCAB_FORGET_SIZE,\n         self.cs.V_OFFSET_MQAR, self.n1.N_KEYS_MQAR, self.n1.N_VALS_MQAR,\n         self.n1.V_OFFSET_MQAR) = self.saved\n\n\ndef build_delta_nlms(vocab_size, max_len, d_model, n_layers):\n    return DeltaNlmsLM(vocab_size, max_len, d_model, n_layers)\n\n\ndef fr_spec(epochs, max_seq_len=112, n_train=600, n_valid=128):\n    from tests.common_smoke import make_task_specs\n    for s in make_task_specs(smoke=\'smoke\'):\n        if s.name == \'forget_retrieval\':\n            s = replace(\n                s, max_seq_len=max_seq_len, epochs=epochs,\n                make_train=lambda seed, n=n_train: ForgetRetrieveDataset(\n                    n, seed=seed, n_pairs_range=(32, 48), n_forget_range=(1, 2)),\n                make_valid=lambda seed, n=n_valid: ForgetRetrieveDataset(\n                    n, seed=seed + 1, n_pairs_range=(32, 48), n_forget_range=(1, 2)))\n            return s\n    raise RuntimeError(\'forget_retrieval spec no encontrada\')\n\n\ndef probe_wave_2x2(model, seed, n_events, device, n_pairs_range=(32, 48)):\n    """Probe 2x2 completo: 4 celdas x 6 delta, fuga pooled + EM_erased\n    con guarda de residual nulo. Autocheck EM_alive vs harness adentro."""\n    out = {}\n    for kind in (\'alive\', \'erased\'):\n        x, y, seqs, v_erased = make_fr_probe_dataset(\n            n_events, seed=1000 + seed, n_pairs_range=n_pairs_range, kind=kind)\n        x = x.to(device)\n        if kind == \'alive\':\n            em, _ = replay_wave(model, x, y, seqs, v_erased, \'clave\', 0.0, device)\n            out[\'em_alive\'] = round(em, 4)\n        else:\n            cells = {}\n            for cell in (\'clave\', \'estado\'):\n                cells[cell] = {}\n                for dlt in DELTAS:\n                    em, fuga, ratios = replay_wave(\n                        model, x, y, seqs, v_erased, cell, dlt, device,\n                        per_event=True)\n                    cells[cell][str(dlt)] = {\n                        \'fuga\': round(fuga[0], 4),\n                        \'pred\': round(law(cell, \'complex\' if not model.read_proj == \'re\' else \'re\',\n                                          dlt, 39.0 / 128.0, \'fiel\'), 4),\n                        **guarded_em(em, fuga, ratios)}\n            out[\'cells\'] = cells\n    return out\n\n\ndef em_probe(model, x, y, seqs, device):\n    """EM exacto del forward del modelo (delta_nlms o wave) con la\n    interfaz estandar: decode_step secuencial, logits en ANSWER."""\n    B, T = x.shape\n    pos_ans = torch.tensor([s.index(ANSWER_ID) for s in seqs], dtype=torch.long,\n                           device=device)\n    state = model.init_state(B, device)\n    pred = torch.zeros(B, dtype=torch.long, device=device)\n    with torch.no_grad():\n        for t in range(T):\n            logits, state = model.decode_step(x[:, t], state)\n            if (pos_ans == t).any():\n                p = pred.clone()\n                p[pos_ans == t] = logits[pos_ans == t].argmax(dim=-1)\n                pred = p\n    return float((pred == y.to(device)).float().mean().item())\n\n\ndef guarded_em(em, fuga, ratios, eps=EPS_NULL):\n    """Guarda de residual nulo (adicion 1 del auditor): EM_erased se\n    reporta solo sobre eventos con ||residual|| > eps*||v_erased||. Si no\n    hay eventos no-nulos -> \'residual nulo\' (resultado mas fuerte que\n    cualquier EM bajo)."""\n    null_frac = float((ratios <= eps).float().mean().item())\n    return {\'em_guarded\': em if null_frac < 1.0 else \'residual nulo\',\n            \'null_frac\': round(null_frac, 4),\n            \'em_on_nonnull_frac\': round(1.0 - null_frac, 4)}\n\n\ndef tost_equivalence(diffs, eps=0.02):\n    n = len(diffs)\n    mu = sum(diffs) / n\n    sd = (sum((d - mu) ** 2 for d in diffs) / max(n - 1, 1)) ** 0.5\n    se = sd / math.sqrt(n)\n    half = T_CRIT_90_DF4 * se\n    return {\'mean_delta\': round(mu, 4), \'sd\': round(sd, 4),\n            \'ci90\': [round(mu - half, 4), round(mu + half, 4)],\n            \'equivalence_pass\': bool(abs(mu) + half <= eps), \'eps\': eps}\n\n\ndef probe_wave_2x2(model, seed, n_events, device, n_pairs_range=(32, 48)):\n    """Probe 2x2 completo: 4 celdas x 6 delta, fuga pooled + EM_erased\n    con guarda de residual nulo. Autocheck EM_alive vs harness adentro."""\n    out = {}\n    for kind in (\'alive\', \'erased\'):\n        x, y, seqs, v_erased = make_fr_probe_dataset(\n            n_events, seed=1000 + seed, n_pairs_range=n_pairs_range, kind=kind)\n        x = x.to(device)\n        if kind == \'alive\':\n            em, _ = replay_wave(model, x, y, seqs, v_erased, \'clave\', 0.0, device)\n            out[\'em_alive\'] = round(em, 4)\n        else:\n            cells = {}\n            for cell in (\'clave\', \'estado\'):\n                cells[cell] = {}\n                for dlt in DELTAS:\n                    em, fuga, ratios = replay_wave(\n                        model, x, y, seqs, v_erased, cell, dlt, device,\n                        per_event=True)\n                    cells[cell][str(dlt)] = {\n                        \'fuga\': round(fuga[0], 4),\n                        \'pred\': round(law(cell, \'complex\' if not model.read_proj == \'re\' else \'re\',\n                                          dlt, 39.0 / 128.0, \'fiel\'), 4),\n                        **guarded_em(em, fuga, ratios)}\n            out[\'cells\'] = cells\n    return out\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--device\', type=str, default=\'cpu\')\n    ap.add_argument(\'--seeds\', type=str, default=\'1,2,3,4,5\')\n    ap.add_argument(\'--epochs\', type=int, default=60)\n    ap.add_argument(\'--delta_epochs\', type=int, default=150)\n    ap.add_argument(\'--delta_train\', type=int, default=1200)\n    ap.add_argument(\'--n_probe\', type=int, default=N_EVENTS)\n    ap.add_argument(\'--d64_dir\', type=str, default=\'outputs\')\n    ap.add_argument(\'--out\', type=str, default=\'outputs/wave_mem/o05.json\')\n    args = ap.parse_args()\n    device = torch.device(args.device)\n    seeds = [int(s) for s in args.seeds.split(\',\')]\n    c = 39.0 / 128.0  # n medio 40 -> c = (n-1)/D\n\n    arms = [(\'wave_complex\', build_wave(\'complex\')),\n            (\'wave_re\', build_wave(\'re\')),\n            (\'delta_nlms\', build_delta_nlms)]\n    with KeySpace(48, 48, 105, 57):\n        spec_wave = fr_spec(args.epochs)\n        spec_delta = fr_spec(args.delta_epochs, n_train=args.delta_train)\n        print(f\'FR d=128: n_pairs [32,48] n_forget [1,2] max_seq {spec_wave.max_seq_len} \'\n              f\'c={c} vocab={CS.VOCAB_FORGET_SIZE}\', flush=True)\n        print(f\'  wave: {spec_wave.epochs} ep x 600 / delta_nlms: \'\n              f\'{spec_delta.epochs} ep x {args.delta_train} (receta n1, \'\n              f\'v4 mostro EM 0.07-0.98 sub-entrenado)\', flush=True)\n\n        t0 = time.time()\n        out = {\'variant\': \'o05\', \'d_model\': 128, \'n_layers\': 2, \'seeds\': seeds,\n               \'n_probe_events\': args.n_probe, \'c\': c,\n               \'spec\': \'FR only, n_pairs [32,48], 48 keys/48 vals (vocab 105, \'\n                       \'KeySpace O05), wave 60ep x 600 / delta_nlms 150ep x 1200 \'\n                       \'(v5; v4: wave EM 1.000 a 60ep x 600, delta sub-entrenado)\',\n               \'eps_null\': EPS_NULL, \'runs\': {}}\n\n        def _ckpt(arm, seed, epochs):\n            return (f\'outputs/o5_{arm}/cache/forget_retrieval_seed{seed}\'\n                    f\'_dm128_L2_ep{epochs}.pt\')\n\n        for seed in seeds:\n            for arm_name, build in arms:\n                spec = spec_delta if arm_name == \'delta_nlms\' else spec_wave\n                label = f\'{arm_name}_s{seed}\'\n                print(f\'\\n===== O05 {arm_name} seed={seed} \'\n                      f\'({spec.epochs}ep x {spec.make_train(0).__len__()}) =====\',\n                      flush=True)\n                ck = _ckpt(arm_name, seed, spec.epochs)\n                if os.path.exists(ck):\n                    ckpt = torch.load(ck, map_location=device)\n                    train_em = ckpt[\'meta\'][\'final\'][\'valid_exact_match\']\n                    print(f\'  (resume) EM={train_em:.3f}\', flush=True)\n                else:\n                    res = run_one_task(spec, build, device, seed=seed, lr=1e-3,\n                                       weight_decay=0.01, clip=1.0,\n                                       d_model=128, n_layers=2,\n                                       variant=f\'o5_{arm_name}\')\n                    train_em = res[\'final_exact_match\']\n                    print(f\'  train EM={train_em:.3f} loss={res["final_loss"]:.3f} \'\n                          f\'t={res["seconds"]:.0f}s\', flush=True)\n                model = build(vocab_size=CS.VOCAB_FORGET_SIZE,\n                              max_len=spec.max_seq_len, d_model=128, n_layers=2)\n                ckpt = torch.load(ck, map_location=device)\n                model.load_state_dict(ckpt[\'model\'])\n                model.to(device).eval()\n                if any(torch.isnan(p).any().item() for p in model.parameters()):\n                    print(\'  WARNING: NaN en ckpt\', flush=True)\n                    out[\'runs\'][label] = {\'train_em\': train_em, \'status\': \'nan\'}\n                    continue\n\n                # Autocheck: EM del probe (alive) vs harness oficial.\n                from functools import partial\n                from torch.utils.data import DataLoader\n                val_ds = ForgetRetrieveDataset(64, seed=seed + 100,\n                                               n_pairs_range=(32, 48),\n                                               n_forget_range=(1, 2))\n                col = partial(collate_fn, answer_marker_id=ANSWER_ID,\n                              mark_after_marker=False, prefix_answer=False)\n                val_loader = DataLoader(val_ds, batch_size=32, shuffle=False,\n                                        collate_fn=col)\n                val_res = evaluate(model, val_loader, device,\n                                   vocab_size=CS.VOCAB_FORGET_SIZE)\n                print(f\'  autocheck harness EM={val_res["exact_match"]:.3f}\',\n                      flush=True)\n\n                if arm_name in (\'wave_complex\', \'wave_re\'):\n                    probe = probe_wave_2x2(model, seed, args.n_probe, device)\n                    probe[\'autocheck_harness_em\'] = round(val_res[\'exact_match\'], 3)\n                    ok = abs(probe[\'em_alive\'] - val_res[\'exact_match\']) < 0.15\n                    probe[\'autocheck_ok\'] = bool(ok)\n                    if not ok:\n                        print(\'  WARNING: probe alive NO matchea harness - abort\',\n                              flush=True)\n                        sys.exit(2)\n                    # EM selectividad (TOST) a d=0 sin drift: probe erased reread.\n                    x, y, seqs, v_erased = make_fr_probe_dataset(\n                        args.n_probe, seed=1000 + seed, n_pairs_range=(32, 48),\n                        kind=\'erased\')\n                    em_e, _, _ = replay_wave(model, x.to(device), y, seqs,\n                                             v_erased, \'clave\', 0.0, device,\n                                             per_event=True)\n                    probe[\'em_erased_d0\'] = round(em_e, 4)\n                    probe[\'selectividad\'] = round(probe[\'em_alive\'] - em_e, 4)\n                    print(f\'  probe: EM_vivo={probe["em_alive"]} \'\n                          f\'sel={probe["selectividad"]}\', flush=True)\n                else:\n                    # delta_nlms: EM alive/erased (selectividad para TOST).\n                    probe = {}\n                    for kind in (\'alive\', \'erased\'):\n                        x, y, seqs, v_erased = make_fr_probe_dataset(\n                            args.n_probe, seed=1000 + seed, n_pairs_range=(32, 48),\n                            kind=kind)\n                        em = em_probe(model, x.to(device), y, seqs, device)\n                        probe[\'em_\' + kind] = round(em, 4)\n                    probe[\'autocheck_harness_em\'] = round(val_res[\'exact_match\'], 3)\n                    ok = abs(probe[\'em_alive\'] - val_res[\'exact_match\']) < 0.15\n                    probe[\'autocheck_ok\'] = bool(ok)\n                    probe[\'selectividad\'] = round(probe[\'em_alive\'] - probe[\'em_erased\'], 4)\n                    print(f\'  probe: EM_vivo={probe["em_alive"]} \'\n                          f\'EM_fuga={probe["em_erased"]} sel={probe["selectividad"]}\',\n                          flush=True)\n                out[\'runs\'][label] = {\'train_em\': round(train_em, 4), \'probe\': probe}\n\n        # ---- TOST (wave_complex vs delta_nlms, selectividad, 5 seeds) ----\n        tost_diffs = []\n        for s in seeds:\n            wc = out[\'runs\'][f\'wave_complex_s{s}\'][\'probe\'].get(\'selectividad\')\n            dl = out[\'runs\'][f\'delta_nlms_s{s}\'][\'probe\'].get(\'selectividad\')\n            if wc is not None and dl is not None:\n                tost_diffs.append(wc - dl)\n        if len(tost_diffs) == len(seeds):\n            out[\'tost\'] = tost_equivalence(tost_diffs)\n        else:\n            out[\'tost\'] = {\'status\': \'NO COMPUTABLE\', \'n_common\': len(tost_diffs)}\n\n    # ---- C1 a c emparejada: probe d=64 con n_pairs (17,24) ----\n    # c = 19.5/64 = 0.3047, el mismo c que d=128. Espacio de claves\n    # ORIGINAL (40/40, vocab 89) fuera del KeySpace.\n    c1 = {}\n    d64 = {\'complex\': f\'{args.d64_dir}/n1_wave_complex/cache\',\n           \'re\': f\'{args.d64_dir}/n1_wave_re/cache\'}\n    for arm, chan in ((\'wave_complex\', \'complex\'), (\'wave_re\', \'re\')):\n        c1[arm] = {}\n        for seed in seeds[:3]:\n            ck = f\'{d64[chan]}/forget_retrieval_seed{seed}_dm64_L2_ep80.pt\'\n            if not os.path.exists(ck):\n                print(f\'  C1: falta {ck} - skip\', flush=True)\n                c1[arm][f\'s{seed}\'] = {\'status\': \'skip\'}\n                continue\n            model = build_wave(chan)(vocab_size=89, max_len=64, d_model=64,\n                                     n_layers=2)\n            model.load_state_dict(torch.load(ck, map_location=device)[\'model\'])\n            model.to(device).eval()\n            c1[arm][f\'s{seed}\'] = {}\n            for cell in (\'clave\', \'estado\'):\n                c1[arm][f\'s{seed}\'][cell] = {}\n                for dlt in DELTAS:\n                    x, y, seqs, v_erased = make_fr_probe_dataset(\n                        args.n_probe, seed=1000 + seed, n_pairs_range=(17, 24),\n                        kind=\'erased\')\n                    em, fuga, _ = replay_wave(model, x.to(device), y, seqs,\n                                              v_erased, cell, dlt, device,\n                                              per_event=True)\n                    c1[arm][f\'s{seed}\'][cell][str(dlt)] = round(fuga[0], 4)\n    # diff d=128 vs d=64 por celda (max sobre delta y seeds 1-3)\n    c1[\'diff_d128_d64\'] = {}\n    for arm in (\'wave_complex\', \'wave_re\'):\n        c1[\'diff_d128_d64\'][arm] = {}\n        for cell in (\'clave\', \'estado\'):\n            mx = 0.0\n            for seed in seeds[:3]:\n                if \'status\' in c1[arm].get(f\'s{seed}\', {}):\n                    continue\n                for dlt in DELTAS:\n                    f128 = out[\'runs\'][f\'{arm}_s{seed}\'][\'probe\'][\'cells\'][cell][str(dlt)][\'fuga\']\n                    f64 = c1[arm][f\'s{seed}\'][cell][str(dlt)]\n                    mx = max(mx, abs(f128 - f64))\n            c1[\'diff_d128_d64\'][arm][cell] = round(mx, 4)\n            c1[\'diff_d128_d64\'][arm][cell + \'_pass\'] = bool(mx < 0.02)\n    out[\'c1_matched_c\'] = c1\n\n    out[\'total_seconds\'] = round(time.time() - t0, 1)\n    os.makedirs(os.path.dirname(args.out), exist_ok=True)\n    with open(args.out, \'w\', encoding=\'utf-8\') as f:\n        json.dump(out, f, indent=2, default=str)\n    print(f\'\\nEscrito: {args.out}  total={out["total_seconds"]}s\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()''')
print('wrote tests/wave_mem_smoke128.py:', os.path.getsize(os.path.join(base, 'tests/wave_mem_smoke128.py')))

# --- prototypes/wave_mem/model.py ---
with open(os.path.join(base, 'prototypes/wave_mem/model.py'), 'w', encoding='utf-8') as f:
    f.write('''"""Standalone wave-memory LM: interference accumulator in C^{DxD}.\n\nMemory per block: M in C^{D x D}. Fixed unit-phasor key codebook (frozen\nbuffer, v1). Structural marker-driven interface (identical to\ndelta_forget): BOS/SEP/FORGET/QUERY => next non-marker is a KEY; after a\nKEY => next non-marker is a VALUE. Writes at VALUE tokens with the pending\nKEY\'s phasor; erases at the KEY following FORGET_ID (alpha=1 fixed, value\nre-read from M); reads at the KEY following QUERY_ID; the readout persists\nuntil ANSWER. No learned gates, no attention, no RoPE.\n"""\n\nimport math\n\nimport torch\nimport torch.nn as nn\n\nMARKER_MIN = 9\nFORGET_ID = 6\nQUERY_ID = 4\n\n\ndef make_codebook(vocab_size: int, d_model: int, seed: int = 0) -> torch.Tensor:\n    g = torch.Generator().manual_seed(seed)\n    theta = torch.rand(vocab_size, d_model, generator=g) * 2.0 * math.pi\n    return torch.complex(torch.cos(theta), torch.sin(theta))\n\n\nclass WaveMemState:\n    __slots__ = (\'M\', \'r\', \'prev\', \'pending_id\', \'expect_value\')\n\n    def __init__(self, M, r, prev, pending_id, expect_value):\n        self.M = M\n        self.r = r\n        self.prev = prev\n        self.pending_id = pending_id\n        self.expect_value = expect_value\n\n\nclass WaveMemBlock(nn.Module):\n    def __init__(self, d_model: int, read_proj: str = \'complex\'):\n        super().__init__()\n        self.d_model = d_model\n        self.read_proj = read_proj\n        self.v_proj = nn.Linear(d_model, d_model)\n        out_dim = 2 * d_model if read_proj == \'complex\' else d_model\n        self.out_proj = nn.Linear(out_dim, d_model)\n\n    def forward(self, x, M, w_cur, w_pending, write, erase, read):\n        # v real (O03, decision pre-registrada): canal imaginario del valor = 0\n        v_c = torch.complex(self.v_proj(x), torch.zeros_like(self.v_proj(x)))\n        B = M.size(0)\n        if write.any() or erase.any():\n            delta = torch.zeros_like(M)\n            if write.any():\n                delta[write] = w_pending[write].unsqueeze(-1) * v_c[write].unsqueeze(1)\n            if erase.any():\n                wh = torch.einsum(\'bd,bdj->bj\', torch.conj(w_cur[erase]), M[erase])\n                wh = wh / self.d_model\n                if self.read_proj == \'re\':\n                    wh = torch.complex(wh.real, torch.zeros_like(wh.real))\n                delta[erase] = -w_cur[erase].unsqueeze(-1) * wh.unsqueeze(1)\n            M = M + delta\n        r = None\n        if read.any():\n            rr = torch.einsum(\'bd,bdj->bj\', torch.conj(w_cur[read]), M[read])\n            rr = rr / self.d_model\n            if self.read_proj == \'re\':\n                rr = torch.complex(rr.real, torch.zeros_like(rr.real))\n            r = (read, rr)\n        return M, r, v_c\n\n\nclass WaveMemLM(nn.Module):\n    def __init__(self, vocab_size: int, max_len: int, d_model: int,\n                 n_layers: int = 2, read_proj: str = \'complex\',\n                 codebook_seed: int = 0):\n        super().__init__()\n        self.vocab_size = int(vocab_size)\n        self.d_model = int(d_model)\n        self.n_layers = int(n_layers)\n        self.read_proj = read_proj\n        self.embedding = nn.Embedding(vocab_size, d_model)\n        self.register_buffer(\'codebook\', make_codebook(vocab_size, d_model, codebook_seed),\n                             persistent=False)\n        self.blocks = nn.ModuleList([WaveMemBlock(d_model, read_proj)\n                                     for _ in range(n_layers)])\n        self.norm = nn.LayerNorm(d_model)\n        self.head = nn.Linear(d_model, vocab_size)\n\n    def init_state(self, batch_size: int, device=None):\n        if device is None:\n            device = self.embedding.weight.device\n        M = [torch.zeros(batch_size, self.d_model, self.d_model,\n                         dtype=torch.complex64, device=device)\n             for _ in range(self.n_layers)]\n        r = [torch.zeros(batch_size, self.d_model, dtype=torch.complex64,\n                         device=device) for _ in range(self.n_layers)]\n        prev = torch.zeros(batch_size, dtype=torch.long, device=device)\n        pending_id = torch.zeros(batch_size, dtype=torch.long, device=device)\n        expect_value = torch.zeros(batch_size, dtype=torch.bool, device=device)\n        return WaveMemState(M, r, prev, pending_id, expect_value)\n\n    def decode_step(self, tok, state):\n        B = tok.size(0)\n        x = self.embedding(tok)\n        tok_is_marker = tok < MARKER_MIN\n        cur_is_key = ~tok_is_marker\n        prev_forget = state.prev == FORGET_ID\n        prev_query = state.prev == QUERY_ID\n        write = state.expect_value & cur_is_key\n        erase = prev_forget & cur_is_key\n        read = prev_query & cur_is_key\n        w_cur = self.codebook[tok]\n        w_pending = self.codebook[state.pending_id]\n        M = state.M\n        r_persist = state.r\n        for b, block in enumerate(self.blocks):\n            M_b = M[b]\n            M_b, r_new, _ = block(x, M_b, w_cur, w_pending, write, erase, read)\n            M[b] = M_b\n            if r_new is not None:\n                ridx, rr = r_new\n                rp = r_persist[b].clone()\n                rp[ridx] = rr\n                r_persist[b] = rp\n        h = x\n        for b, block in enumerate(self.blocks):\n            rb = r_persist[b]\n            if self.read_proj == \'complex\':\n                feat = torch.cat([rb.real, rb.imag], dim=-1)\n            else:\n                feat = rb.real\n            h = h + block.out_proj(feat)\n        logits = self.head(self.norm(h))\n        state.prev = tok\n        state.expect_value = ~tok_is_marker & ~state.expect_value\n        state.pending_id = torch.where(cur_is_key & ~write, tok, state.pending_id)\n        return logits, state\n\n    def forward(self, input_ids):\n        B, T = input_ids.shape\n        if T == 0:\n            return torch.empty(B, 0, self.vocab_size, device=input_ids.device,\n                               dtype=self.embedding.weight.dtype)\n        state = self.init_state(B, input_ids.device)\n        outs = []\n        for t in range(T):\n            logits_t, state = self.decode_step(input_ids[:, t], state)\n            outs.append(logits_t)\n        return torch.stack(outs, dim=1)\n\n    def prefill(self, input_ids, state=None):\n        B, T = input_ids.shape\n        if state is None:\n            state = self.init_state(B, input_ids.device)\n        outs = []\n        for t in range(T):\n            logits_t, state = self.decode_step(input_ids[:, t], state)\n            outs.append(logits_t)\n        if outs:\n            logits = torch.stack(outs, dim=1)\n        else:\n            logits = torch.empty(B, 0, self.vocab_size, device=input_ids.device,\n                                 dtype=self.embedding.weight.dtype)\n        return logits, state''')
print('wrote prototypes/wave_mem/model.py:', os.path.getsize(os.path.join(base, 'prototypes/wave_mem/model.py')))

# --- prototypes/delta_forget/model.py ---
with open(os.path.join(base, 'prototypes/delta_forget/model.py'), 'w', encoding='utf-8') as f:
    f.write('''"""Standalone delta-rule LM: sequential delta rule in R^{DxD}.\n\nComparator arm for wave_mem with an IDENTICAL structural marker interface:\nBOS/SEP/FORGET/QUERY => next non-marker is a KEY; after a KEY => next\nnon-marker is a VALUE. Writes at VALUE tokens binding the pending KEY\'s\nprojection (K(emb_key)) to the value projection (V(emb_value)); erases at\nthe KEY following FORGET_ID with beta=1 forced (projection (I - uu^T),\nu = k/||k||); reads at the KEY following QUERY_ID with the learned q; the\nreadout persists until ANSWER. No conv1d, no RoPE, no learned erase\namplitude, no learned write gate (beta=1 fixed): the operator is the only\ndifference vs wave_mem (key channel and read projection).\n"""\n\nimport torch\nimport torch.nn as nn\n\nMARKER_MIN = 9\nFORGET_ID = 6\nQUERY_ID = 4\n\n\nclass DeltaState:\n    __slots__ = (\'S\', \'r\', \'prev\', \'pending_id\', \'expect_value\')\n\n    def __init__(self, S, r, prev, pending_id, expect_value):\n        self.S = S\n        self.r = r\n        self.prev = prev\n        self.pending_id = pending_id\n        self.expect_value = expect_value\n\n\nclass DeltaBlock(nn.Module):\n    def __init__(self, d_model: int):\n        super().__init__()\n        self.d_model = d_model\n        self.k_proj = nn.Linear(d_model, d_model)\n        self.v_proj = nn.Linear(d_model, d_model)\n        self.q_proj = nn.Linear(d_model, d_model)\n        self.out_proj = nn.Linear(d_model, d_model)\n\n    def forward(self, x, x_pending, S, write, erase, read):\n        k_cur = self.k_proj(x)\n        if write.any() or erase.any():\n            delta = torch.zeros_like(S)\n            if write.any():\n                k_pend = self.k_proj(x_pending[write])\n                v = self.v_proj(x[write])\n                kS = torch.einsum(\'bd,bdj->bj\', k_pend, S[write])\n                delta[write] = (v - kS).unsqueeze(1) * k_pend.unsqueeze(2)\n            if erase.any():\n                k_e = self.k_proj(x[erase])\n                kS = torch.einsum(\'bd,bdj->bj\', k_e, S[erase])\n                n2 = (k_e * k_e).sum(dim=1).clamp(min=1e-8).unsqueeze(1)\n                delta[erase] = -(kS / n2).unsqueeze(1) * k_e.unsqueeze(2)\n            S = S + delta\n        r = None\n        if read.any():\n            q = self.q_proj(x[read])\n            r = (read, torch.einsum(\'bd,bdj->bj\', q, S[read]))\n        return S, r, k_cur\n\n\nclass DeltaForgetLM(nn.Module):\n    def __init__(self, vocab_size: int, max_len: int, d_model: int,\n                 n_layers: int = 2):\n        super().__init__()\n        self.vocab_size = int(vocab_size)\n        self.d_model = int(d_model)\n        self.n_layers = int(n_layers)\n        self.embedding = nn.Embedding(vocab_size, d_model)\n        self.blocks = nn.ModuleList([DeltaBlock(d_model) for _ in range(n_layers)])\n        self.norm = nn.LayerNorm(d_model)\n        self.head = nn.Linear(d_model, vocab_size)\n\n    def init_state(self, batch_size: int, device=None):\n        if device is None:\n            device = self.embedding.weight.device\n        S = [torch.zeros(batch_size, self.d_model, self.d_model, device=device)\n             for _ in range(self.n_layers)]\n        r = [torch.zeros(batch_size, self.d_model, device=device)\n             for _ in range(self.n_layers)]\n        prev = torch.zeros(batch_size, dtype=torch.long, device=device)\n        pending_id = torch.zeros(batch_size, dtype=torch.long, device=device)\n        expect_value = torch.zeros(batch_size, dtype=torch.bool, device=device)\n        return DeltaState(S, r, prev, pending_id, expect_value)\n\n    def decode_step(self, tok, state):\n        x = self.embedding(tok)\n        tok_is_marker = tok < MARKER_MIN\n        cur_is_key = ~tok_is_marker\n        prev_forget = state.prev == FORGET_ID\n        prev_query = state.prev == QUERY_ID\n        write = state.expect_value & cur_is_key\n        erase = prev_forget & cur_is_key\n        read = prev_query & cur_is_key\n        x_pending = self.embedding(state.pending_id)\n        S = state.S\n        r_persist = state.r\n        for b, block in enumerate(self.blocks):\n            S_b = S[b]\n            S_b, r_new, _ = block(x, x_pending, S_b, write, erase, read)\n            S[b] = S_b\n            if r_new is not None:\n                ridx, rr = r_new\n                rp = r_persist[b].clone()\n                rp[ridx] = rr\n                r_persist[b] = rp\n        h = x\n        for b, block in enumerate(self.blocks):\n            h = h + block.out_proj(r_persist[b])\n        logits = self.head(self.norm(h))\n        state.prev = tok\n        state.expect_value = ~tok_is_marker & ~state.expect_value\n        state.pending_id = torch.where(cur_is_key & ~write, tok, state.pending_id)\n        return logits, state\n\n    def forward(self, input_ids):\n        B, T = input_ids.shape\n        if T == 0:\n            return torch.empty(B, 0, self.vocab_size, device=input_ids.device,\n                               dtype=self.embedding.weight.dtype)\n        state = self.init_state(B, input_ids.device)\n        outs = []\n        for t in range(T):\n            logits_t, state = self.decode_step(input_ids[:, t], state)\n            outs.append(logits_t)\n        return torch.stack(outs, dim=1)\n\n    def prefill(self, input_ids, state=None):\n        B, T = input_ids.shape\n        if state is None:\n            state = self.init_state(B, input_ids.device)\n        outs = []\n        for t in range(T):\n            logits_t, state = self.decode_step(input_ids[:, t], state)\n            outs.append(logits_t)\n        if outs:\n            logits = torch.stack(outs, dim=1)\n        else:\n            logits = torch.empty(B, 0, self.vocab_size, device=input_ids.device,\n                                 dtype=self.embedding.weight.dtype)\n        return logits, state''')
print('wrote prototypes/delta_forget/model.py:', os.path.getsize(os.path.join(base, 'prototypes/delta_forget/model.py')))

# --- prototypes/delta_nlms/model.py ---
with open(os.path.join(base, 'prototypes/delta_nlms/model.py'), 'w', encoding='utf-8') as f:
    f.write('''"""Standalone NLMS-delta-rule LM (autopsia O03-N1, side arm del auditor):\nmismo delta_rule real en R^{DxD} que delta_forget pero con CLAVES\nNORMALIZADAS (k/||k||, NLMS de Widrow-Hoff) y beta=1 fijo.\n\nLa autopsia (tests/delta_autopsy*.py) probo que el write correctivo de\ndelta_forget (beta=1, claves sin normalizar) es un lazo cerrado con\nganancia |1 - beta||k||^2|: con ||k||^2 >> 1 la memoria S diverge\n(||S|| ~ 1e12 en una pasada) y la task no entrena. Con ||k|| = 1 la\nganancia es 0: estable por construccion, erase = proyeccion exacta con\nclave unitaria, sin gates aprendidas. Si entrena forget_retrieval, el\ncontrol TOST (identidad wave_complex vs delta) RESUCITA; si no, la\ndeclaracion "delta FALLIDO en FR" queda hermeticamente documentada.\n\nMisma interfaz estructural de markers (BOS/SEP/FORGET/QUERY => next\nnon-marker KEY; tras KEY => VALUE). Escribe en VALUE con k_pend\nnormalizado; borra en el KEY tras FORGET_ID con proyeccion de clave\nunitaria; lee en el KEY tras QUERY_ID con q normalizado; el readout\npersiste hasta ANSWER. Sin gates aprendidas, sin atencion, sin RoPE.\n"""\n\nimport torch\nimport torch.nn as nn\n\nMARKER_MIN = 9\nFORGET_ID = 6\nQUERY_ID = 4\n\n\nclass NlmsState:\n    __slots__ = (\'S\', \'r\', \'prev\', \'pending_id\', \'expect_value\')\n\n    def __init__(self, S, r, prev, pending_id, expect_value):\n        self.S = S\n        self.r = r\n        self.prev = prev\n        self.pending_id = pending_id\n        self.expect_value = expect_value\n\n\nclass NlmsBlock(nn.Module):\n    def __init__(self, d_model: int):\n        super().__init__()\n        self.d_model = d_model\n        self.k_proj = nn.Linear(d_model, d_model)\n        self.v_proj = nn.Linear(d_model, d_model)\n        self.q_proj = nn.Linear(d_model, d_model)\n        self.out_proj = nn.Linear(d_model, d_model)\n\n    def _norm(self, u):\n        n = u.norm(dim=1, keepdim=True).clamp(min=1e-8)\n        return u / n\n\n    def forward(self, x, x_pending, S, write, erase, read):\n        k_cur = self._norm(self.k_proj(x))\n        if write.any() or erase.any():\n            delta = torch.zeros_like(S)\n            if write.any():\n                k_pend = self._norm(self.k_proj(x_pending[write]))\n                v = self.v_proj(x[write])\n                kS = torch.einsum(\'bd,bdj->bj\', k_pend, S[write])\n                delta[write] = (v - kS).unsqueeze(1) * k_pend.unsqueeze(2)\n            if erase.any():\n                k_e = self._norm(self.k_proj(x[erase]))\n                kS = torch.einsum(\'bd,bdj->bj\', k_e, S[erase])\n                delta[erase] = -(kS).unsqueeze(1) * k_e.unsqueeze(2)\n            S = S + delta\n        r = None\n        if read.any():\n            q = self._norm(self.q_proj(x[read]))\n            r = (read, torch.einsum(\'bd,bdj->bj\', q, S[read]))\n        return S, r, k_cur\n\n\nclass DeltaNlmsLM(nn.Module):\n    def __init__(self, vocab_size: int, max_len: int, d_model: int,\n                 n_layers: int = 2):\n        super().__init__()\n        self.vocab_size = int(vocab_size)\n        self.d_model = int(d_model)\n        self.n_layers = int(n_layers)\n        self.embedding = nn.Embedding(vocab_size, d_model)\n        self.blocks = nn.ModuleList([NlmsBlock(d_model) for _ in range(n_layers)])\n        self.norm = nn.LayerNorm(d_model)\n        self.head = nn.Linear(d_model, vocab_size)\n\n    def init_state(self, batch_size: int, device=None):\n        if device is None:\n            device = self.embedding.weight.device\n        S = [torch.zeros(batch_size, self.d_model, self.d_model, device=device)\n             for _ in range(self.n_layers)]\n        r = [torch.zeros(batch_size, self.d_model, device=device)\n             for _ in range(self.n_layers)]\n        prev = torch.zeros(batch_size, dtype=torch.long, device=device)\n        pending_id = torch.zeros(batch_size, dtype=torch.long, device=device)\n        expect_value = torch.zeros(batch_size, dtype=torch.bool, device=device)\n        return NlmsState(S, r, prev, pending_id, expect_value)\n\n    def decode_step(self, tok, state):\n        x = self.embedding(tok)\n        tok_is_marker = tok < MARKER_MIN\n        cur_is_key = ~tok_is_marker\n        prev_forget = state.prev == FORGET_ID\n        prev_query = state.prev == QUERY_ID\n        write = state.expect_value & cur_is_key\n        erase = prev_forget & cur_is_key\n        read = prev_query & cur_is_key\n        x_pending = self.embedding(state.pending_id)\n        S = state.S\n        r_persist = state.r\n        for b, block in enumerate(self.blocks):\n            S_b = S[b]\n            S_b, r_new, _ = block(x, x_pending, S_b, write, erase, read)\n            S[b] = S_b\n            if r_new is not None:\n                ridx, rr = r_new\n                rp = r_persist[b].clone()\n                rp[ridx] = rr\n                r_persist[b] = rp\n        h = x\n        for b, block in enumerate(self.blocks):\n            h = h + block.out_proj(r_persist[b])\n        logits = self.head(self.norm(h))\n        state.prev = tok\n        state.expect_value = ~tok_is_marker & ~state.expect_value\n        state.pending_id = torch.where(cur_is_key & ~write, tok, state.pending_id)\n        return logits, state\n\n    def forward(self, input_ids):\n        B, T = input_ids.shape\n        if T == 0:\n            return torch.empty(B, 0, self.vocab_size, device=input_ids.device,\n                               dtype=self.embedding.weight.dtype)\n        state = self.init_state(B, input_ids.device)\n        outs = []\n        for t in range(T):\n            logits_t, state = self.decode_step(input_ids[:, t], state)\n            outs.append(logits_t)\n        return torch.stack(outs, dim=1)\n\n    def prefill(self, input_ids, state=None):\n        B, T = input_ids.shape\n        if state is None:\n            state = self.init_state(B, input_ids.device)\n        outs = []\n        for t in range(T):\n            logits_t, state = self.decode_step(input_ids[:, t], state)\n            outs.append(logits_t)\n        if outs:\n            logits = torch.stack(outs, dim=1)\n        else:\n            logits = torch.empty(B, 0, self.vocab_size, device=input_ids.device,\n                                 dtype=self.embedding.weight.dtype)\n        return logits, state''')
print('wrote prototypes/delta_nlms/model.py:', os.path.getsize(os.path.join(base, 'prototypes/delta_nlms/model.py')))

# --- tests/__init__.py ---
with open(os.path.join(base, 'tests/__init__.py'), 'w', encoding='utf-8') as f:
    f.write('')

# --- prototypes/__init__.py ---
with open(os.path.join(base, 'prototypes/__init__.py'), 'w', encoding='utf-8') as f:
    f.write('')


In [ ]:
# ==== CKPTS d=64 (C1) desde dataset ====
import glob
import shutil

hits = [p for p in glob.glob('/kaggle/input/**/forget_retrieval_seed1_dm64_L2_ep80.pt', recursive=True)
        if '/complex/' in p]
if hits:
    shutil.copy(hits[0], os.path.join(base, 'outputs/n1_wave_complex/cache/forget_retrieval_seed1_dm64_L2_ep80.pt'))
    print('CKPT OK: cache/complex/forget_retrieval_seed1_dm64_L2_ep80.pt')
else:
    print('CKPT FALTA: cache/complex/forget_retrieval_seed1_dm64_L2_ep80.pt')

hits = [p for p in glob.glob('/kaggle/input/**/forget_retrieval_seed2_dm64_L2_ep80.pt', recursive=True)
        if '/complex/' in p]
if hits:
    shutil.copy(hits[0], os.path.join(base, 'outputs/n1_wave_complex/cache/forget_retrieval_seed2_dm64_L2_ep80.pt'))
    print('CKPT OK: cache/complex/forget_retrieval_seed2_dm64_L2_ep80.pt')
else:
    print('CKPT FALTA: cache/complex/forget_retrieval_seed2_dm64_L2_ep80.pt')

hits = [p for p in glob.glob('/kaggle/input/**/forget_retrieval_seed3_dm64_L2_ep80.pt', recursive=True)
        if '/complex/' in p]
if hits:
    shutil.copy(hits[0], os.path.join(base, 'outputs/n1_wave_complex/cache/forget_retrieval_seed3_dm64_L2_ep80.pt'))
    print('CKPT OK: cache/complex/forget_retrieval_seed3_dm64_L2_ep80.pt')
else:
    print('CKPT FALTA: cache/complex/forget_retrieval_seed3_dm64_L2_ep80.pt')

hits = [p for p in glob.glob('/kaggle/input/**/forget_retrieval_seed1_dm64_L2_ep80.pt', recursive=True)
        if '/re/' in p]
if hits:
    shutil.copy(hits[0], os.path.join(base, 'outputs/n1_wave_re/cache/forget_retrieval_seed1_dm64_L2_ep80.pt'))
    print('CKPT OK: cache/re/forget_retrieval_seed1_dm64_L2_ep80.pt')
else:
    print('CKPT FALTA: cache/re/forget_retrieval_seed1_dm64_L2_ep80.pt')

hits = [p for p in glob.glob('/kaggle/input/**/forget_retrieval_seed2_dm64_L2_ep80.pt', recursive=True)
        if '/re/' in p]
if hits:
    shutil.copy(hits[0], os.path.join(base, 'outputs/n1_wave_re/cache/forget_retrieval_seed2_dm64_L2_ep80.pt'))
    print('CKPT OK: cache/re/forget_retrieval_seed2_dm64_L2_ep80.pt')
else:
    print('CKPT FALTA: cache/re/forget_retrieval_seed2_dm64_L2_ep80.pt')

hits = [p for p in glob.glob('/kaggle/input/**/forget_retrieval_seed3_dm64_L2_ep80.pt', recursive=True)
        if '/re/' in p]
if hits:
    shutil.copy(hits[0], os.path.join(base, 'outputs/n1_wave_re/cache/forget_retrieval_seed3_dm64_L2_ep80.pt'))
    print('CKPT OK: cache/re/forget_retrieval_seed3_dm64_L2_ep80.pt')
else:
    print('CKPT FALTA: cache/re/forget_retrieval_seed3_dm64_L2_ep80.pt')


In [ ]:
import subprocess, sys, os
import torch
n_gpu = torch.cuda.device_count()
print('GPUs:', n_gpu)
procs = []
if n_gpu >= 2:
    halves = [('1,2,3', 'o05_part0.json'), ('4,5', 'o05_part1.json')]
    for i, (seeds, out_name) in enumerate(halves):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        procs.append(subprocess.Popen(
            [sys.executable, 'tests/wave_mem_smoke128.py',
             '--device', 'cuda', '--seeds', seeds,
             '--out', 'outputs/wave_mem/' + out_name],
            cwd='/kaggle/working/onda', env=env))
    rcs = [p.wait() for p in procs]
    print('workers rc =', rcs)
    assert all(r == 0 for r in rcs), 'worker fallo'
else:
    rc = subprocess.call([sys.executable, 'tests/wave_mem_smoke128.py',
                          '--device', 'cuda'], cwd='/kaggle/working/onda')
    print('runner rc =', rc)
    assert rc == 0, 'runner fallo'
    import shutil
    shutil.copy('outputs/wave_mem/o05.json', 'outputs/wave_mem/o05_part0.json')


In [ ]:
import json, math
base = '/kaggle/working/onda/outputs/wave_mem'
if os.path.exists(base + '/o05_part1.json'):
    p0 = json.load(open(base + '/o05_part0.json'))
    p1 = json.load(open(base + '/o05_part1.json'))
    runs = dict(p0['runs'])
    runs.update(p1['runs'])
    final = dict(p0)
    final['runs'] = runs
    diffs = [runs['wave_complex_s%d' % s]['probe']['selectividad']
             - runs['delta_nlms_s%d' % s]['probe']['selectividad']
             for s in (1, 2, 3, 4, 5)]
    mu = sum(diffs) / 5
    sd = (sum((d - mu) ** 2 for d in diffs) / 4) ** 0.5
    half = 2.132 * sd / math.sqrt(5)
    final['tost'] = {'mean_delta': round(mu, 4), 'sd': round(sd, 4),
                     'ci90': [round(mu - half, 4), round(mu + half, 4)],
                     'equivalence_pass': bool(abs(mu) + half <= 0.02),
                     'eps': 0.02, 'n_seeds': 5}
    final['total_seconds'] = max(p0['total_seconds'], p1['total_seconds'])
    json.dump(final, open(base + '/o05.json', 'w'), indent=2)
    print('MERGE OK')
else:
    print('secuencial, o05.json ya final')


In [ ]:
import json
p = '/kaggle/working/onda/outputs/wave_mem/o05.json'
d = json.load(open(p))
print('TOST:', json.dumps(d.get('tost'), indent=1))
print('C1 diff:', json.dumps(d.get('c1_matched_c', {}).get('diff_d128_d64', {}), indent=1))
print('runs:', {k: v.get('train_em') for k, v in d.get('runs', {}).items()})
print('total_s:', d.get('total_seconds'))
